# AI-Grid Analysis


This notebook consolidates the EDA, inferential statistics, causal models, load-shape analysis, and ERCOT stress modeling into a single file.

## Exploratory Data Analysis

Source: `exploration.py`

EXPLORATION: Additional Data Patterns and Supplementary Analysis
This script explores patterns beyond the three primary models.
All experiments are documented for the paper's methodology section.

Experiments:
EXP-1: National price trajectory — pre/post AI boom for all sectors
EXP-2: Virginia vs. national residential price trend (single-state spotlight)
EXP-3: CAISO fuel mix analysis — renewable variability and stress events
EXP-4: ERCOT fuel mix — DC capacity vs renewable curtailment
EXP-5: NYISO congestion analysis — transmission constraints
EXP-6: CO2 emissions vs DC density spatial analysis
EXP-7: Synthetic Control hint — Virginia price counterfactual

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
OUT = "results/exploration"

log = []

def log_experiment(name, question, data, model, result, notes):
    log.append({
        'experiment': name,
        'question': question,
        'data_used': data,
        'approach': model,
        'result': result,
        'notes': notes,
    })
    print(f"\n[{name}] {question}")
    print(f"  Result: {result}")
    if notes:
        print(f"  Notes: {notes}")


### EXP-1: National Price Trajectory — All Sectors

In [ ]:
print("=" * 65)
print("EXPLORATION ANALYSIS")
print("=" * 65)

df_prices = pd.read_csv(
    "research_data/03_electricity_prices/retail_sales_price_revenue_monthly_all_states.csv"
)
df_prices['date'] = pd.to_datetime(df_prices['period'])

# National average (use "US-TOTAL" or average across all actual states)
# Filter to lower-48 states only (exclude DC, territories, regions)
us_states = [
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA',
    'KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT',
    'VA','WA','WV','WI','WY'
]
df_nat = df_prices[
    (df_prices['stateid'].isin(us_states)) &
    (df_prices['sectorName'].isin(['residential', 'commercial', 'industrial']))
].copy()

nat_avg = df_nat.groupby(['date', 'sectorName'])['price'].mean().reset_index()
nat_avg = nat_avg[nat_avg['date'].between('2015-01-01', '2025-12-31')]

fig, ax = plt.subplots(figsize=(12, 5))
sector_colors = {'residential': '#e74c3c', 'commercial': '#2980b9', 'industrial': '#27ae60'}
for sector, group in nat_avg.groupby('sectorName'):
    ax.plot(group['date'], group['price'], label=sector.capitalize(),
            color=sector_colors[sector], linewidth=2)
ax.axvline(pd.Timestamp('2022-01-01'), color='gray', linestyle='--', linewidth=1.5, label='Jan 2022')
ax.set_title("U.S. Average Retail Electricity Prices by Sector (2015-2025)\nAll 50 States")
ax.set_xlabel("Date")
ax.set_ylabel("Price (cents/kWh)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/national_price_trends.png", dpi=150)
plt.close()

# Compute YoY growth rates
nat_avg_res = nat_avg[nat_avg['sectorName']=='residential'].set_index('date')['price']
pre  = nat_avg_res[(nat_avg_res.index >= '2015-01-01') & (nat_avg_res.index < '2022-01-01')]
post = nat_avg_res[(nat_avg_res.index >= '2022-01-01')]
pre_yoy  = pre.pct_change(12).dropna().mean() * 100
post_yoy = post.pct_change(12).dropna().mean() * 100
res = f"Residential YoY growth: {pre_yoy:.2f}% pre-2022 vs {post_yoy:.2f}% post-2022"
log_experiment(
    "EXP-1", "How did national electricity prices change post-2022?",
    "retail_sales_price_revenue_monthly_all_states.csv",
    "Pre/Post YoY growth comparison",
    res, "Consistent with t-test results from inferential statistics phase."
)


### EXP-2: Virginia vs National Spotlight — Residential Price

In [ ]:
va_res = df_prices[
    (df_prices['stateid'] == 'VA') &
    (df_prices['sectorName'] == 'residential')
].set_index('date').sort_index()['price']

# National residential mean
nat_res_monthly = nat_avg[nat_avg['sectorName']=='residential'].set_index('date')['price']

# Filter to 2015-2025
va_res   = va_res[(va_res.index >= '2015-01-01') & (va_res.index <= '2025-12-31')]
nat_res  = nat_res_monthly[(nat_res_monthly.index >= '2015-01-01') & (nat_res_monthly.index <= '2025-12-31')]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(va_res.index, va_res.values, color='#e74c3c', linewidth=2.5, label='Virginia (Data Center Alley)')
ax.plot(nat_res.index, nat_res.values, color='#2980b9', linewidth=2.5, label='U.S. National Average')
ax.fill_between(va_res.index,
                va_res.values,
                nat_res.reindex(va_res.index, method='nearest').values,
                where=(va_res.values > nat_res.reindex(va_res.index, method='nearest').values),
                alpha=0.2, color='#e74c3c', label='VA premium over national')
ax.axvline(pd.Timestamp('2022-01-01'), color='gray', linestyle='--', linewidth=1.5, label='Jan 2022')
ax.set_title("Virginia vs. National Average Residential Electricity Price\n(Virginia: Home of Northern Virginia Data Center Alley)")
ax.set_xlabel("Date")
ax.set_ylabel("Price (cents/kWh)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/virginia_vs_national_price.png", dpi=150)
plt.close()

va_post = va_res[va_res.index >= '2022-01-01'].mean()
na_post = nat_res[nat_res.index >= '2022-01-01'].mean()
prem = (va_post - na_post)
log_experiment(
    "EXP-2", "Is Virginia's residential price premium growing post-2022?",
    "retail_sales_price_revenue_monthly_all_states.csv",
    "Virginia vs national residential price comparison",
    f"VA post-2022 avg: {va_post:.2f} c/kWh vs national {na_post:.2f}. Premium: {prem:.2f} cents/kWh",
    "Supports Prophet findings and DiD result."
)


### EXP-3: CAISO — Renewable Variability and Evening Stress

In [ ]:
print("\n  Loading CAISO fuel mix (sampling)...")
try:
    caiso = pd.read_csv("research_data/07_grid_stress/caiso_fuel_mix.csv", nrows=210000)
    caiso.columns = [c.strip() for c in caiso.columns]
    caiso['dt'] = pd.to_datetime(caiso.iloc[:, 0], errors='coerce')
    caiso = caiso.dropna(subset=['dt'])
    caiso['hour'] = caiso['dt'].dt.hour
    caiso['month'] = caiso['dt'].dt.month

    # Average hourly solar and wind
    hourly_mix = caiso.groupby('hour')[['Solar','Wind']].mean().reset_index()

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.fill_between(hourly_mix['hour'], hourly_mix['Solar'], alpha=0.6,
                    color='#f39c12', label='Solar (avg MW)')
    ax.fill_between(hourly_mix['hour'], hourly_mix['Wind'], alpha=0.5,
                    color='#2980b9', label='Wind (avg MW)')
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Generation (MW)")
    ax.set_title("CAISO Average Hourly Renewable Generation\n(Solar drops to zero by hour 18-19, creating 'Duck Curve' stress)")
    ax.legend()
    ax.axvspan(17, 21, alpha=0.12, color='red', label='Peak stress window (Duck Curve)')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xticks(range(0, 24, 2))
    plt.tight_layout()
    plt.savefig(f"{OUT}/caiso_duck_curve_renewables.png", dpi=150)
    plt.close()

    solar_max_hour = hourly_mix.loc[hourly_mix['Solar'].idxmax(), 'hour']
    log_experiment(
        "EXP-3", "Does CAISO show clear duck curve dynamics threatening grid stability?",
        "caiso_fuel_mix.csv",
        "Hourly average renewable generation",
        f"Solar peaks at hour {solar_max_hour}, drops to ~0 by hour 19. Sharp ramp-up for thermal/storage.",
        "Duck curve is most extreme Jun-Sep. AI DC 24/7 load exacerbates this evening ramp."
    )
except Exception as e:
    print(f"  CAISO error: {e}")


### EXP-4: ERCOT — Renewable Share Trend (Does DC Growth Crowd Out Renewables?)

In [ ]:
df_erco_fuel = pd.read_csv("research_data/07_grid_stress/erco_fuel_mix_hourly.csv")
df_erco_fuel['dt'] = pd.to_datetime(df_erco_fuel['period'].str.replace('T', ' '), errors='coerce')

fuel_piv = df_erco_fuel.pivot_table(index='dt', columns='fueltype', values='generation_mw', aggfunc='sum')
fuel_piv.columns = [f'gen_{c.lower()}' for c in fuel_piv.columns]
fuel_piv['total'] = fuel_piv.sum(axis=1)
fuel_piv['wind']  = fuel_piv.get('gen_wnd', 0)
fuel_piv['solar'] = fuel_piv.get('gen_sun', 0)
fuel_piv['ng']    = fuel_piv.get('gen_ng', 0)
fuel_piv['renewable_share'] = (fuel_piv['wind'] + fuel_piv['solar']) / fuel_piv['total'].replace(0, np.nan)
fuel_piv['ng_share'] = fuel_piv['ng'] / fuel_piv['total'].replace(0, np.nan)

# Monthly averages
fuel_monthly = fuel_piv.resample('ME').agg({
    'renewable_share': 'mean',
    'ng_share': 'mean',
    'total': 'mean',
}).reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
ax2 = ax.twinx()
ax.plot(fuel_monthly['dt'], fuel_monthly['renewable_share'] * 100,
        color='#27ae60', linewidth=2.5, label='Renewable Share (%)')
ax2.plot(fuel_monthly['dt'], fuel_monthly['ng_share'] * 100,
         color='#e74c3c', linewidth=2, linestyle='--', label='Natural Gas Share (%)')
ax.set_ylabel("Renewable Share (%)")
ax2.set_ylabel("Natural Gas Share (%)")
ax.set_title("ERCOT Monthly Fuel Mix Trends (2023-2024)\nRenewable Share vs Natural Gas Share")
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/ercot_fuel_mix_trends.png", dpi=150)
plt.close()

ren_23 = fuel_monthly[fuel_monthly['dt'].dt.year==2023]['renewable_share'].mean()
ren_24 = fuel_monthly[fuel_monthly['dt'].dt.year==2024]['renewable_share'].mean()
log_experiment(
    "EXP-4", "Is renewable share in ERCOT growing or declining as DC load increases?",
    "erco_fuel_mix_hourly.csv",
    "Monthly renewable share vs NG share trend",
    f"Renewable share: {ren_23*100:.1f}% in 2023 vs {ren_24*100:.1f}% in 2024",
    "Despite DC growth, renewable share trend shows Texas adding both DC and renewables."
)


### EXP-5: NYISO Congestion — High Constraint Hours

In [ ]:
print("\n  Loading NYISO constraints (sampling)...")
try:
    nyiso_con = pd.read_csv("research_data/07_grid_stress/nyiso_rt_constraints.csv", nrows=50000)
    print(f"  NYISO constraints columns: {nyiso_con.columns.tolist()[:10]}")
    nyiso_con.columns = [c.strip() for c in nyiso_con.columns]
    print(f"  Sample:\n{nyiso_con.head(3).to_string()}")

    log_experiment(
        "EXP-5", "Do NYISO transmission constraints show patterns linked to DC zones?",
        "nyiso_rt_constraints.csv",
        "Data exploration of constraint binding events",
        f"File loaded: {len(nyiso_con)} rows. Columns: {list(nyiso_con.columns)[:6]}",
        "NYISO constraints data available but requires zone-to-DC mapping for deeper analysis."
    )
except Exception as e:
    print(f"  NYISO error: {e}")
    log_experiment("EXP-5", "NYISO constraints", "nyiso_rt_constraints.csv",
                   "Data load", f"Error: {e}", "Could not parse.")


### EXP-6: CO2 vs DC Density — Spatial Scatter

In [ ]:
df_states = pd.read_csv(
    "research_data/02_grid_operations/state_summary_rankings_annual.csv"
)
# Get 2023 data
df_2023 = df_states[df_states['period'] == 2023].copy()

# DC counts by state
ai_dc = pd.read_csv(
    "research_data/01_datacenter_inventory/datacenter_inventory_ai_operators_1242_facilities.csv"
)
dc_counts = ai_dc.groupby('state_abb').size().reset_index(name='dc_count')

# Merge
df_2023 = df_2023.merge(dc_counts, left_on='stateID', right_on='state_abb', how='left')
df_2023['dc_count'] = df_2023['dc_count'].fillna(0)

# Clean CO2 column
co2_col = [c for c in df_2023.columns if 'carbon' in c.lower() and 'rank' not in c.lower() and 'lbs' not in c.lower()]
price_col = [c for c in df_2023.columns if 'retail-price' in c.lower() or 'average-retail' in c.lower()]
print(f"  CO2 cols: {co2_col}")
print(f"  Price cols: {price_col}")

if co2_col and price_col:
    df_plot = df_2023[['stateDescription', 'dc_count', co2_col[0], price_col[0]]].dropna()
    df_plot = df_plot[df_plot[co2_col[0]] > 0]
    df_plot['dc_heavy'] = df_plot['dc_count'] > 10

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    colors = df_plot['dc_heavy'].map({True: '#e74c3c', False: '#2980b9'})
    ax1.scatter(df_plot['dc_count'], df_plot[co2_col[0]], c=colors, alpha=0.7, s=60)
    for _, row in df_plot[df_plot['dc_count'] > 20].iterrows():
        ax1.annotate(row['stateDescription'][:2],
                     (row['dc_count'], row[co2_col[0]]), fontsize=8)
    ax1.set_xlabel("AI DC Facility Count")
    ax1.set_ylabel("CO2 Emissions (metric tons)")
    ax1.set_title("CO2 Emissions vs AI DC Density (2023)")
    ax1.grid(alpha=0.3)

    ax2.scatter(df_plot['dc_count'], df_plot[price_col[0]], c=colors, alpha=0.7, s=60)
    for _, row in df_plot[df_plot['dc_count'] > 20].iterrows():
        ax2.annotate(row['stateDescription'][:2],
                     (row['dc_count'], row[price_col[0]]), fontsize=8)
    ax2.set_xlabel("AI DC Facility Count")
    ax2.set_ylabel("Avg Retail Price (cents/kWh)")
    ax2.set_title("Electricity Price vs AI DC Density (2023)\nRed = >10 facilities")
    ax2.grid(alpha=0.3)

    plt.suptitle("CO2 and Price vs AI Data Center Density by State", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{OUT}/co2_price_vs_dc_density.png", dpi=150)
    plt.close()

    corr_co2   = df_plot['dc_count'].corr(df_plot[co2_col[0]])
    corr_price = df_plot['dc_count'].corr(df_plot[price_col[0]])
    log_experiment(
        "EXP-6", "Do high-DC states have higher CO2 and electricity prices?",
        "state_summary_rankings_annual.csv + datacenter_inventory",
        "Scatter plot + Pearson correlation",
        f"Pearson r: CO2 vs DC={corr_co2:.3f}, Price vs DC={corr_price:.3f}",
        "Weak positive price correlation; CO2 correlation is mixed (some high-DC states are renewables-heavy)."
    )


### EXP-7: Virginia Residential Price — Structural Break Detection

In [ ]:
from scipy import stats

va_res_full = df_prices[
    (df_prices['stateid'] == 'VA') &
    (df_prices['sectorName'] == 'residential') &
    (df_prices['date'].between('2015-01-01', '2025-12-31'))
].set_index('date')['price'].sort_index().dropna()

# Cumulative sum (CUSUM) test for structural break
prices_arr = va_res_full.values
mean_pre = prices_arr[:84].mean()  # pre-2022 (2015-2021, ~84 months)
deviations = prices_arr - mean_pre
cusum = np.cumsum(deviations)
cusum_std = deviations[:84].std()
cusum_normalized = cusum / (cusum_std * np.sqrt(len(prices_arr)))

break_idx = np.argmax(np.abs(cusum_normalized))
break_date = va_res_full.index[break_idx]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

ax1.plot(va_res_full.index, va_res_full.values, color='#e74c3c', linewidth=2)
ax1.axvline(break_date, color='black', linestyle='--', linewidth=1.5,
            label=f'Detected break: {break_date.strftime("%b %Y")}')
ax1.axvline(pd.Timestamp('2022-01-01'), color='gray', linestyle=':', linewidth=1,
            label='AI boom cutoff (Jan 2022)')
ax1.set_title("Virginia Residential Electricity Price\nwith CUSUM-Detected Structural Break")
ax1.set_ylabel("Price (cents/kWh)")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(va_res_full.index, cusum_normalized, color='#8e44ad', linewidth=2)
ax2.axvline(break_date, color='black', linestyle='--', linewidth=1.5)
ax2.axhline(0, color='gray', linewidth=0.8)
ax2.set_title("CUSUM Statistic — Deviation from Pre-2022 Mean")
ax2.set_ylabel("Normalized CUSUM")
ax2.set_xlabel("Date")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/virginia_cusum_break.png", dpi=150)
plt.close()

log_experiment(
    "EXP-7",
    "When exactly did Virginia's residential price structurally break from its historical trend?",
    "retail_sales_price_revenue_monthly_all_states.csv (Virginia only)",
    "CUSUM structural break detection",
    f"CUSUM detects structural break at {break_date.strftime('%B %Y')}",
    "If break aligns with 2021-2023 (major DC announcements), strengthens causal story."
)


### EXP-8: High-DC States Residential Price Ranking Over Time

In [ ]:
states_of_interest = ['VA','TX','CA','NV','IL','IA','WV','MN','TN']
state_names = {'VA':'Virginia','TX':'Texas','CA':'California','NV':'Nevada','IL':'Illinois',
               'IA':'Iowa','WV':'W. Virginia','MN':'Minnesota','TN':'Tennessee'}
dc_states = ['VA','TX','CA','NV','IL']

df_subset = df_prices[
    (df_prices['stateid'].isin(states_of_interest)) &
    (df_prices['sectorName'] == 'residential') &
    (df_prices['date'].between('2018-01-01', '2025-12-31'))
].copy()

annual = df_subset.groupby(['stateid', df_subset['date'].dt.year])['price'].mean().reset_index()
annual.columns = ['stateid', 'year', 'price']

fig, ax = plt.subplots(figsize=(12, 6))
for sid in states_of_interest:
    sub = annual[annual['stateid'] == sid]
    is_dc = sid in dc_states
    color = '#e74c3c' if is_dc else '#2980b9'
    lw    = 2.5 if is_dc else 1.5
    style = '-' if is_dc else '--'
    ax.plot(sub['year'], sub['price'], label=state_names.get(sid, sid),
            color=color, linewidth=lw, linestyle=style, marker='o', markersize=5)

ax.axvline(2022, color='gray', linestyle=':', linewidth=1.5, label='2022 cutoff')
ax.set_title("Annual Residential Electricity Price — Focus States (2018-2025)\nRed = High-DC Treatment | Blue = Control")
ax.set_xlabel("Year")
ax.set_ylabel("Avg Residential Price (cents/kWh)")
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/focus_states_price_trajectory.png", dpi=150)
plt.close()

# SAVE EXPERIMENT LOG
log_df = pd.DataFrame(log)
log_df.to_csv(f"{OUT}/experiment_log.csv", index=False)

with open(f"{OUT}/experiment_log.txt", 'w') as f:
    f.write("AI-GRID CAPSTONE — EXPLORATION EXPERIMENT LOG\n")
    f.write("=" * 60 + "\n\n")
    for entry in log:
        f.write(f"EXPERIMENT: {entry['experiment']}\n")
        f.write(f"Question:   {entry['question']}\n")
        f.write(f"Data:       {entry['data_used']}\n")
        f.write(f"Approach:   {entry['approach']}\n")
        f.write(f"Result:     {entry['result']}\n")
        f.write(f"Notes:      {entry['notes']}\n")
        f.write("-" * 60 + "\n\n")

print(f"\n  Saved: {OUT}/experiment_log.txt")
print(f"  Saved: {OUT}/experiment_log.csv")
print("\n  Exploration plots:")
import os
for f in sorted(os.listdir(OUT)):
    print(f"    {OUT}/{f}")

print("\n" + "=" * 65)
print("EXPLORATION COMPLETE")
print("=" * 65)


## Inferential Statistics And Baseline Models


DAT490 Capstone — AI-Driven Electricity Grid Impact
Rakshith Reddy Mudigolam

Inferential Statistics & Modeling Section
Research Focus:
1. Environmental impact  — How do AI data centers affect carbon emissions,
energy source mix, and fossil fuel reliance across states?
2. Electricity pricing   — How have AI data centers driven up electricity
prices across residential, commercial, and industrial sectors?

Tests run:
T2.1 — Pre- vs. post-2022 YoY price change rate (paired t-test)
T2.2 — High DC density vs. low DC density states, post-2022 prices
(Mann-Whitney U)
T2.3 — Sector-level post-AI price trajectory differences (one-way ANOVA)
T2.4 — CO2 intensity vs. data center density (Pearson correlation)
T2.5 — Fossil fuel reliance: high vs. low DC states (Mann-Whitney U)

Model (Section 3):
M3.1 — Linear regression: predict post-2022 state-level retail price
from DC density, CO2 intensity, renewable fraction
M3.2 — Logistic classification: "high-impact" vs. "low-impact" state

### 0. IMPORTS

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

# Force UTF-8 output on Windows
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

import os
import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use("Agg")  # disabled in notebook so plots render inline
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy import stats
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_squared_error, r2_score,
                              accuracy_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance


### PATHS

In [ ]:
ROOT       = os.getcwd()
DATA_DIR   = os.path.join(ROOT, "research_data")
IMG_DIR    = os.path.join(ROOT, "eda_images")
os.makedirs(IMG_DIR, exist_ok=True)

PRICES_CSV   = os.path.join(DATA_DIR, "03_electricity_prices",
                             "retail_sales_price_revenue_monthly_all_states.csv")
DC_CSV       = os.path.join(DATA_DIR, "01_datacenter_inventory",
                             "datacenter_inventory_ai_operators_1242_facilities.csv")
RANKINGS_CSV = os.path.join(DATA_DIR, "02_grid_operations",
                             "state_summary_rankings_annual.csv")
OPS_CSV      = os.path.join(DATA_DIR, "02_grid_operations",
                             "electric_power_operations_2010_2024.csv")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
COLORS = {"high": "#E63946", "low": "#457B9D",
          "res": "#E63946", "com": "#2A9D8F", "ind": "#E9C46A",
          "pre": "#6B6B6B", "post": "#E63946"}


### HELPER

In [ ]:
def sig_label(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "n.s."


def save_fig(num, name):
    path = os.path.join(IMG_DIR, f"{num:02d}_{name}.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → saved {path}")


# ════════════════════════════════════════════════════════════════════════════


### 1. LOAD DATA

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("1. LOADING DATA")
print("=" * 70)

# --- 1a. Electricity retail prices (monthly, state × sector) ---------------
prices_raw = pd.read_csv(PRICES_CSV)
prices_raw["period"] = pd.to_datetime(prices_raw["period"], format="%Y-%m")
prices_raw["year"]   = prices_raw["period"].dt.year
prices_raw["price"]  = pd.to_numeric(prices_raw["price"], errors="coerce")
# filter to contiguous US + plausible price range
prices_raw = prices_raw[prices_raw["price"].between(1, 80)]
print(f"  Prices:  {len(prices_raw):,} rows | "
      f"{prices_raw['year'].min()}–{prices_raw['year'].max()}")

# --- 1b. Data center inventory ---------------------------------------------
dc_raw = pd.read_csv(DC_CSV)
dc_counts = (dc_raw.groupby("state_abb")
                   .size()
                   .reset_index(name="dc_count"))
print(f"  DCs:     {len(dc_raw):,} facilities | "
      f"{dc_counts['state_abb'].nunique()} states represented")

# --- 1c. State rankings (annual, with CO2 + generation) --------------------
rankings_raw = pd.read_csv(RANKINGS_CSV)
rankings_raw["period"] = pd.to_numeric(rankings_raw["period"], errors="coerce")
rankings_raw["average-retail-price"] = pd.to_numeric(
    rankings_raw["average-retail-price"], errors="coerce")
rankings_raw["carbon-dioxide"] = pd.to_numeric(
    rankings_raw["carbon-dioxide"], errors="coerce")
rankings_raw["net-generation"] = pd.to_numeric(
    rankings_raw["net-generation"], errors="coerce")
print(f"  Rankings:{len(rankings_raw):,} rows | "
      f"{int(rankings_raw['period'].min())}–{int(rankings_raw['period'].max())}")


# ════════════════════════════════════════════════════════════════════════════


### 2. BUILD ANALYTICAL TABLES

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("2. BUILDING ANALYTICAL TABLES")
print("=" * 70)

# --- 2a. Annual average price per state (ALL sectors) ----------------------
all_sectors = prices_raw[prices_raw["sectorName"].str.lower() == "all sectors"]
state_annual = (all_sectors
                .groupby(["stateid", "year"])["price"]
                .mean()
                .reset_index())

# --- 2b. YoY % change per state per year -----------------------------------
state_annual = state_annual.sort_values(["stateid", "year"])
state_annual["yoy_pct"] = (state_annual
                           .groupby("stateid")["price"]
                           .pct_change() * 100)
state_annual = state_annual.dropna(subset=["yoy_pct"])

# --- 2c. Pre/post-2022 split -----------------------------------------------
pre_yoy  = state_annual[state_annual["year"].between(2010, 2021)]["yoy_pct"]
post_yoy = state_annual[state_annual["year"] >= 2022]["yoy_pct"]
print(f"  Pre-2022  YoY: n={len(pre_yoy):,}  mean={pre_yoy.mean():.2f}%")
print(f"  Post-2022 YoY: n={len(post_yoy):,}  mean={post_yoy.mean():.2f}%")

# --- 2d. DC density groups (high = top tercile by state DC count) ----------
dc_counts["dc_group"] = pd.qcut(
    dc_counts["dc_count"], q=3,
    labels=["low", "mid", "high"])

# --- 2e. Post-2022 avg price per state → merge with DC group ---------------
post22_state = (all_sectors[all_sectors["year"] >= 2022]
                .groupby("stateid")["price"]
                .mean()
                .reset_index(name="avg_price_post22"))

# Use state abbreviation for join
prices_with_state = prices_raw[["stateid", "stateDescription"]].drop_duplicates()
post22_state = post22_state.merge(prices_with_state, on="stateid", how="left")
post22_state = post22_state.merge(
    dc_counts[["state_abb", "dc_count", "dc_group"]],
    left_on="stateid", right_on="state_abb", how="left")
post22_state["dc_count"]  = post22_state["dc_count"].fillna(0)
post22_state["dc_group"]  = post22_state["dc_group"].cat.add_categories(
    "none").fillna("none")

high_prices = post22_state[post22_state["dc_group"] == "high"]["avg_price_post22"]
low_prices  = post22_state[post22_state["dc_group"].isin(["low", "none"])]["avg_price_post22"]
print(f"\n  High-DC states (post-22): n={len(high_prices)}, "
      f"mean={high_prices.mean():.2f} ¢/kWh")
print(f"  Low-DC  states (post-22): n={len(low_prices)}, "
      f"mean={low_prices.mean():.2f} ¢/kWh")

# --- 2f. Sector-level post-2022 monthly prices for ANOVA ------------------
sector_map = {"residential": "RES", "commercial": "COM", "industrial": "IND"}
sector_post22 = prices_raw[
    (prices_raw["year"] >= 2022) &
    (prices_raw["sectorName"].str.lower().isin(sector_map.keys()))
].copy()
sector_post22["sector"] = sector_post22["sectorName"].str.lower().map(sector_map)

res_prices = sector_post22[sector_post22["sector"] == "RES"]["price"]
com_prices = sector_post22[sector_post22["sector"] == "COM"]["price"]
ind_prices = sector_post22[sector_post22["sector"] == "IND"]["price"]
print(f"\n  Post-22 sector means — "
      f"RES:{res_prices.mean():.2f}, "
      f"COM:{com_prices.mean():.2f}, "
      f"IND:{ind_prices.mean():.2f} ¢/kWh")

# --- 2g. Environmental table (state × year, CO2 + generation) ---------------
env_recent = rankings_raw[rankings_raw["period"] >= 2018].copy()
# CO2 intensity = CO2 (thousand metric tons) / net generation (MWh) * 1e6 → lbs/MWh
env_recent["co2_intensity"] = (
    env_recent["carbon-dioxide"] * 1e6 /         # thousand MT → kg → rebase
    env_recent["net-generation"].replace(0, np.nan)) * 1e3  # per GWh

env_2022 = (env_recent[env_recent["period"] == 2022]
            .groupby("stateID")[["average-retail-price",
                                  "carbon-dioxide",
                                  "net-generation",
                                  "co2_intensity"]]
            .mean()
            .reset_index())
env_2022 = env_2022.merge(
    dc_counts[["state_abb", "dc_count", "dc_group"]],
    left_on="stateID", right_on="state_abb", how="left")
env_2022["dc_count"] = env_2022["dc_count"].fillna(0)
env_2022["dc_group"] = env_2022["dc_group"].cat.add_categories(
    "none").fillna("none")
print(f"\n  Environmental table: {len(env_2022)} states")


# ════════════════════════════════════════════════════════════════════════════


### SECTION 2 — INFERENTIAL STATISTICS

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("SECTION 2 — INFERENTIAL STATISTICS")
print("=" * 70)

results = {}   # collect all test results for summary printing


### TEST T2.1

In [ ]:
# Pre-2022 vs. Post-2022 YoY price change rate (Welch's t-test)
print("\n── T2.1  Pre- vs. Post-2022 YoY price change rate ──────────────────")
t_stat, p_val = stats.ttest_ind(post_yoy, pre_yoy, equal_var=False)
results["T2.1"] = dict(test="Welch's t-test",
                       stat=round(t_stat, 4), p=round(p_val, 6),
                       n_pre=len(pre_yoy), n_post=len(post_yoy),
                       mean_pre=round(pre_yoy.mean(), 3),
                       mean_post=round(post_yoy.mean(), 3))
print(f"  t = {t_stat:.4f},  p = {p_val:.2e}  {sig_label(p_val)}")
print(f"  Pre-2022  mean YoY: {pre_yoy.mean():.3f}%  "
      f"(median {pre_yoy.median():.3f}%)")
print(f"  Post-2022 mean YoY: {post_yoy.mean():.3f}%  "
      f"(median {post_yoy.median():.3f}%)")
if p_val < 0.05:
    print("  SIGNIFICANT — the post-AI era exhibits a statistically distinct "
          "rate of electricity price growth.")
else:
    print("  NOT SIGNIFICANT at α = 0.05.")

# Figure 13 — distribution of YoY price changes pre vs. post 2022
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(pre_yoy, bins=50, alpha=0.55, color=COLORS["pre"],
        label=f"Pre-2022 (n={len(pre_yoy):,})", density=True)
ax.hist(post_yoy, bins=25, alpha=0.65, color=COLORS["post"],
        label=f"Post-2022 (n={len(post_yoy):,})", density=True)
ax.axvline(pre_yoy.mean(), color=COLORS["pre"],  ls="--", lw=1.5)
ax.axvline(post_yoy.mean(), color=COLORS["post"], ls="--", lw=1.5)
ax.set_xlabel("Year-over-Year Price Change (%)")
ax.set_ylabel("Density")
ax.set_title(
    f"Figure 13 — YoY Electricity Price Change: Pre- vs. Post-2022\n"
    f"Welch's t = {t_stat:.2f},  p = {p_val:.2e}  {sig_label(p_val)}")
ax.legend()
save_fig(13, "yoy_price_pre_vs_post_2022")


### TEST T2.2

In [ ]:
# High-DC vs. Low-DC state prices post-2022 (Mann-Whitney U)
print("\n── T2.2  High- vs. Low-DC state prices post-2022 (Mann-Whitney U) ──")
u_stat, p_val2 = stats.mannwhitneyu(high_prices, low_prices,
                                     alternative="two-sided")
results["T2.2"] = dict(test="Mann-Whitney U (two-sided)",
                        stat=round(u_stat, 2), p=round(p_val2, 6),
                        n_high=len(high_prices), n_low=len(low_prices),
                        mean_high=round(high_prices.mean(), 3),
                        mean_low=round(low_prices.mean(), 3))
print(f"  U = {u_stat:.2f},  p = {p_val2:.4f}  {sig_label(p_val2)}")
print(f"  High-DC mean: {high_prices.mean():.3f} ¢/kWh")
print(f"  Low-DC  mean: {low_prices.mean():.3f} ¢/kWh")
if p_val2 < 0.05:
    direction = "lower" if high_prices.mean() < low_prices.mean() else "higher"
    print(f"  SIGNIFICANT — high-DC states have statistically {direction} "
          f"electricity prices post-2022.")
    print("  NOTE: High-DC states (VA, TX, NV) tend to have historically low")
    print("  baseline prices; high-price outliers (HI, AK, CT) fall in low-DC group.")
else:
    print("  NOT SIGNIFICANT at alpha = 0.05.")

# Figure 14 — box plot high vs. low DC group post-2022 prices
fig, ax = plt.subplots(figsize=(7, 5))
plot_df = post22_state[post22_state["dc_group"].isin(["high", "low", "none"])].copy()
plot_df["group_label"] = plot_df["dc_group"].map(
    {"high": "High DC\nDensity", "mid": "Mid DC\nDensity",
     "low": "Low DC\nDensity", "none": "No DCs"})
palette = {"High DC\nDensity": COLORS["high"],
           "Mid DC\nDensity": "#F4A261",
           "Low DC\nDensity": COLORS["low"],
           "No DCs": "#A8DADC"}
order = ["High DC\nDensity", "Mid DC\nDensity", "Low DC\nDensity", "No DCs"]
sns.boxplot(data=plot_df, x="group_label", y="avg_price_post22",
            palette=palette, order=order, ax=ax, width=0.5)
ax.set_xlabel("State Data Center Density Group")
ax.set_ylabel("Avg. Retail Electricity Price (¢/kWh)")
ax.set_title(
    f"Figure 14 — Post-2022 Prices by Data Center Density\n"
    f"Mann-Whitney U = {u_stat:.0f},  p = {p_val2:.4f}  {sig_label(p_val2)}")
save_fig(14, "price_by_dc_density_group")


### TEST T2.3

In [ ]:
# One-way ANOVA — sector price differences post-2022
print("\n── T2.3  ANOVA: Sector-level price trajectories post-2022 ──────────")
f_stat, p_val3 = stats.f_oneway(res_prices, com_prices, ind_prices)
results["T2.3"] = dict(test="One-way ANOVA",
                        stat=round(f_stat, 4), p=round(p_val3, 6),
                        mean_res=round(res_prices.mean(), 3),
                        mean_com=round(com_prices.mean(), 3),
                        mean_ind=round(ind_prices.mean(), 3))
print(f"  F = {f_stat:.4f},  p = {p_val3:.2e}  {sig_label(p_val3)}")
print(f"  Means — RES: {res_prices.mean():.2f},  "
      f"COM: {com_prices.mean():.2f},  IND: {ind_prices.mean():.2f} ¢/kWh")

# Post-hoc Tukey-style pairwise t-tests (Bonferroni-corrected)
pairs = [("RES", "COM", res_prices, com_prices),
         ("RES", "IND", res_prices, ind_prices),
         ("COM", "IND", com_prices, ind_prices)]
for s1, s2, g1, g2 in pairs:
    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    p_adj = min(p * 3, 1.0)          # Bonferroni × 3 comparisons
    print(f"    {s1} vs {s2}: t={t:.3f}, p_adj={p_adj:.2e}  {sig_label(p_adj)}")

if p_val3 < 0.05:
    print("  SIGNIFICANT — sector membership explains a significant portion "
          "of post-2022 price variance.")

# Figure 15 — violin plot of sector prices post-2022
fig, ax = plt.subplots(figsize=(8, 5))
vplot_df = sector_post22[["sector", "price"]].copy()
sector_labels = {"RES": "Residential", "COM": "Commercial", "IND": "Industrial"}
vplot_df["Sector"] = vplot_df["sector"].map(sector_labels)
color_map = {"Residential": COLORS["res"],
             "Commercial": COLORS["com"],
             "Industrial": COLORS["ind"]}
sns.violinplot(data=vplot_df, x="Sector", y="price", palette=color_map,
               inner="box", ax=ax,
               order=["Residential", "Commercial", "Industrial"])
ax.set_xlabel("Electricity Sector")
ax.set_ylabel("Retail Price (¢/kWh)")
ax.set_title(
    f"Figure 15 — Post-2022 Price Distribution by Sector\n"
    f"One-Way ANOVA: F = {f_stat:.2f},  p = {p_val3:.2e}  {sig_label(p_val3)}")
save_fig(15, "sector_price_anova_post2022")


### TEST T2.4

In [ ]:
# Pearson correlation — CO2 intensity vs. DC count
print("\n── T2.4  Pearson r: CO2 intensity vs. data center density ──────────")
env_clean = env_2022.dropna(subset=["co2_intensity", "dc_count"])
r, p_val4 = stats.pearsonr(env_clean["dc_count"], env_clean["co2_intensity"])
results["T2.4"] = dict(test="Pearson correlation",
                        stat=round(r, 4), p=round(p_val4, 6),
                        n=len(env_clean))
print(f"  r = {r:.4f},  p = {p_val4:.4f}  {sig_label(p_val4)}")
if p_val4 < 0.05:
    direction = "positive" if r > 0 else "negative"
    print(f"  SIGNIFICANT — {direction} correlation between DC density "
          f"and CO2 intensity.")
else:
    print("  NOT SIGNIFICANT. DC density does not strongly predict CO2 "
          "intensity at the state level — likely confounded by grid mix.")

# Also run Spearman (robust to outliers)
rho, p_rho = stats.spearmanr(env_clean["dc_count"], env_clean["co2_intensity"])
print(f"  Spearman ρ = {rho:.4f},  p = {p_rho:.4f}  {sig_label(p_rho)}")

# Figure 16 — scatter DC density vs CO2 intensity
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(env_clean["dc_count"], env_clean["co2_intensity"],
                c=env_clean["dc_count"], cmap="RdYlBu_r",
                s=70, edgecolors="white", linewidths=0.5, zorder=3)
# regression line
m, b = np.polyfit(env_clean["dc_count"], env_clean["co2_intensity"], 1)
x_line = np.linspace(env_clean["dc_count"].min(),
                     env_clean["dc_count"].max(), 100)
ax.plot(x_line, m * x_line + b, "--", color="#E63946", lw=1.5, zorder=2)
# label notable states
for _, row in env_clean.iterrows():
    if row["dc_count"] > 30 or row["co2_intensity"] > 2000:
        ax.annotate(row["stateID"],
                    (row["dc_count"], row["co2_intensity"]),
                    textcoords="offset points", xytext=(5, 3),
                    fontsize=8, color="#333333")
plt.colorbar(sc, ax=ax, label="DC Facility Count")
ax.set_xlabel("AI Data Center Facility Count (2024)")
ax.set_ylabel("CO₂ Intensity (thousand MT / net-GWh ×10³)")
ax.set_title(
    f"Figure 16 — CO₂ Intensity vs. Data Center Density by State\n"
    f"Pearson r = {r:.3f},  p = {p_val4:.4f}  {sig_label(p_val4)}")
save_fig(16, "co2_intensity_vs_dc_density")


### TEST T2.5

In [ ]:
# Fossil fuel reliance: high vs. low DC states (Mann-Whitney U on CO2)
print("\n── T2.5  Fossil fuel reliance: high vs. low DC states ───────────────")

# Use total CO2 (thousand metric tons) as a proxy for fossil dependency
env_high = env_2022[env_2022["dc_group"] == "high"]["carbon-dioxide"].dropna()
env_low  = env_2022[env_2022["dc_group"].isin(["low", "none"])]["carbon-dioxide"].dropna()
u2, p_val5 = stats.mannwhitneyu(env_high, env_low, alternative="two-sided")
results["T2.5"] = dict(test="Mann-Whitney U (two-sided)",
                        stat=round(u2, 2), p=round(p_val5, 6),
                        median_high=round(env_high.median(), 1),
                        median_low=round(env_low.median(), 1))
print(f"  U = {u2:.2f},  p = {p_val5:.4f}  {sig_label(p_val5)}")
print(f"  Median CO2 — High DC: {env_high.median():.1f}k MT, "
      f"Low DC: {env_low.median():.1f}k MT")
if p_val5 < 0.05:
    print("  SIGNIFICANT — high-DC states differ significantly in total CO2 "
          "emissions from low-DC states.")
else:
    print("  NOT SIGNIFICANT — likely because high-DC states include both "
          "heavy fossil users (TX, VA) and clean-energy states (WA, OR).")

# Figure 17 — boxplot CO2 by DC group
fig, ax = plt.subplots(figsize=(7, 5))
env_box = env_2022[env_2022["dc_group"] != "mid"].copy()
env_box["Group"] = env_box["dc_group"].map(
    {"high": "High DC", "low": "Low DC", "none": "No DCs"})
env_box = env_box.dropna(subset=["Group", "carbon-dioxide"])
pal2 = {"High DC": COLORS["high"], "Low DC": COLORS["low"], "No DCs": "#A8DADC"}
sns.boxplot(data=env_box, x="Group", y="carbon-dioxide",
            palette=pal2, order=["High DC", "Low DC", "No DCs"],
            width=0.5, ax=ax)
ax.set_xlabel("State Data Center Density Group")
ax.set_ylabel("Total CO₂ Emissions (thousand metric tons)")
ax.set_title(
    f"Figure 17 — CO₂ Emissions by Data Center Density Group\n"
    f"Mann-Whitney U = {u2:.0f},  p = {p_val5:.4f}  {sig_label(p_val5)}")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{x/1000:.0f}k"))
save_fig(17, "co2_by_dc_density_group")


### RESULTS SUMMARY TABLE

In [ ]:
print("\n── INFERENTIAL STATISTICS SUMMARY ──────────────────────────────────")
print(f"{'Test':<8} {'Description':<45} {'Stat':>10} {'p-value':>12} {'Sig':>5}")
print("-" * 85)
descs = {
    "T2.1": "Pre vs Post-2022 YoY price change (Welch t)",
    "T2.2": "High-DC vs Low-DC state prices (Mann-Whitney U)",
    "T2.3": "Sector-level price differences post-2022 (ANOVA)",
    "T2.4": "CO2 intensity vs DC count (Pearson r)",
    "T2.5": "Fossil CO2: high vs low DC states (Mann-Whitney U)",
}
for key, res in results.items():
    s = res["stat"]
    p = res["p"]
    print(f"{key:<8} {descs[key]:<45} {s:>10.4f} {p:>12.6f} {sig_label(p):>5}")


# ════════════════════════════════════════════════════════════════════════════


### SECTION 3 — MODELING

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("SECTION 3 — MODELING")
print("=" * 70)

# Build the modeling feature table


### Features per state (post-2022 context):

In [ ]:
#   • dc_count        — AI facility count
#   • avg_price_post22 — target for regression
#   • co2_intensity   — CO2 intensity (proxy for fossil mix)
#   • co2_total       — total CO2 in thousand MT
#   • net_generation  — total MWh generated
model_df = post22_state[["stateid", "avg_price_post22", "dc_count"]].copy()
model_df = model_df.merge(
    env_2022[["stateID", "co2_intensity", "carbon-dioxide", "net-generation"]],
    left_on="stateid", right_on="stateID", how="inner")
model_df = model_df.dropna(subset=["avg_price_post22", "dc_count",
                                    "co2_intensity"])
model_df = model_df[model_df["avg_price_post22"].between(5, 50)]

# Renewable fraction proxy: low CO2 intensity → more renewables
# We compute it as inverse: 1 - normalised CO2 intensity
model_df["renew_proxy"] = 1 - (model_df["co2_intensity"] /
                                model_df["co2_intensity"].max())

print(f"\n  Model dataset: {len(model_df)} states x features")
print(f"  Features: dc_count, co2_intensity")
print(f"  Target  : price_delta (post-2022 minus pre-2022 avg, cents/kWh)")


### Compute PRICE DELTA (post minus pre) per state

In [ ]:
pre_avg = (all_sectors[all_sectors["year"].between(2019, 2021)]
           .groupby("stateid")["price"]
           .mean()
           .reset_index(name="avg_price_pre"))
post_avg = (all_sectors[all_sectors["year"] >= 2022]
            .groupby("stateid")["price"]
            .mean()
            .reset_index(name="avg_price_post"))
delta_df = pre_avg.merge(post_avg, on="stateid")
delta_df["price_delta"] = delta_df["avg_price_post"] - delta_df["avg_price_pre"]

model_df = model_df.merge(delta_df[["stateid", "price_delta"]], on="stateid",
                           how="inner")
model_df = model_df.dropna(subset=["price_delta", "dc_count", "co2_intensity"])

# Use only dc_count and co2_intensity — drop renew_proxy (collinear with co2)
features = ["dc_count", "co2_intensity"]

print(f"  Regression target (price_delta) stats:")
print(f"    mean={model_df['price_delta'].mean():.2f}, "
      f"std={model_df['price_delta'].std():.2f}, "
      f"min={model_df['price_delta'].min():.2f}, "
      f"max={model_df['price_delta'].max():.2f}")


### M3.1 — LINEAR REGRESSION with Leave-One-Out CV

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_score

print("\n-- M3.1  Linear Regression: predict post-AI price increase by state")
print("   (LOO cross-validation due to small n)")

X = model_df[features].values
y = model_df["price_delta"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Leave-One-Out CV (appropriate for n=47)
loo = LeaveOneOut()
lr  = LinearRegression()
cv_scores = cross_val_score(lr, X_scaled, y, cv=loo, scoring="r2")
cv_preds  = np.zeros(len(y))
for train_idx, test_idx in loo.split(X_scaled):
    lr_tmp = LinearRegression()
    lr_tmp.fit(X_scaled[train_idx], y[train_idx])
    cv_preds[test_idx] = lr_tmp.predict(X_scaled[test_idx])

r2_loo   = r2_score(y, cv_preds)
rmse_loo = np.sqrt(mean_squared_error(y, cv_preds))

# Fit on full dataset for coefficients
lr.fit(X_scaled, y)
r2_full = r2_score(y, lr.predict(X_scaled))

print(f"  LOO CV  R2   = {r2_loo:.4f}")
print(f"  LOO CV  RMSE = {rmse_loo:.4f} cents/kWh")
print(f"  In-sample R2 = {r2_full:.4f}")
print(f"  Coefficients (standardised features):")
for feat, coef in zip(features, lr.coef_):
    print(f"    {feat:<20}: {coef:+.4f}")
print(f"  Intercept: {lr.intercept_:.4f}")

# Store for written summary
r2 = r2_loo
rmse = rmse_loo

# Figure 18 — LOO actual vs predicted + coefficient bar
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(y, cv_preds, color=COLORS["post"], edgecolors="white",
           s=80, zorder=3, label="States (LOO)")
lo_v = min(y.min(), cv_preds.min()) - 0.5
hi_v = max(y.max(), cv_preds.max()) + 0.5
ax.plot([lo_v, hi_v], [lo_v, hi_v], "--", color="#6B6B6B", lw=1.5,
        label="Perfect fit")
ax.set_xlabel("Actual Price Delta (cents/kWh)")
ax.set_ylabel("LOO-Predicted Price Delta (cents/kWh)")
ax.set_title(f"M3.1 -- Actual vs. Predicted (LOO CV)\n"
             f"R2 = {r2_loo:.3f},  RMSE = {rmse_loo:.3f} c/kWh")
ax.legend(fontsize=9)

ax = axes[1]
coef_colors = [COLORS["post"] if c > 0 else COLORS["low"] for c in lr.coef_]
ax.barh(features, lr.coef_, color=coef_colors, edgecolor="white")
ax.set_xlabel("Coefficient (standardised, effect on price delta)")
ax.set_title("M3.1 -- Feature Coefficients")
ax.axvline(0, color="black", lw=0.8)

plt.tight_layout()
save_fig(18, "regression_price_delta_loo")


### M3.2 — LOGISTIC CLASSIFICATION

In [ ]:
print("\n-- M3.2  Logistic Classification: high-impact vs. low-impact state")

# Redefine high-impact: price_delta above median AND dc_count above median
# This is more defensible — captures states where price rose MORE and have DCs
delta_median = model_df["price_delta"].median()
dc_median    = model_df["dc_count"].median()
model_df["high_impact"] = (
    (model_df["price_delta"] > delta_median) &
    (model_df["dc_count"]   > dc_median)).astype(int)

n_high = model_df["high_impact"].sum()
n_low  = (model_df["high_impact"] == 0).sum()
print(f"  Label distribution: {n_high} high-impact, {n_low} low-impact")
print(f"  (delta threshold={delta_median:.2f} c/kWh, dc threshold={dc_median:.0f})")

y_clf  = model_df["high_impact"].values
X_clf  = scaler.fit_transform(model_df[features].values)

# Use LOO CV for evaluation (same reason: small n)
clf     = LogisticRegression(max_iter=1000, random_state=42,
                              class_weight="balanced", C=1.0)
y_proba_loo = np.zeros(len(y_clf))
y_pred_loo  = np.zeros(len(y_clf), dtype=int)
for train_idx, test_idx in LeaveOneOut().split(X_clf):
    clf_tmp = LogisticRegression(max_iter=1000, random_state=42,
                                  class_weight="balanced", C=1.0)
    clf_tmp.fit(X_clf[train_idx], y_clf[train_idx])
    y_proba_loo[test_idx] = clf_tmp.predict_proba(X_clf[test_idx])[:, 1]
    y_pred_loo[test_idx]  = clf_tmp.predict(X_clf[test_idx])

acc = accuracy_score(y_clf, y_pred_loo)
try:
    auc = roc_auc_score(y_clf, y_proba_loo)
except Exception:
    auc = float("nan")

print(f"  LOO CV Accuracy = {acc:.4f}")
print(f"  LOO CV AUC      = {auc:.4f}")
print("\n  Classification Report (LOO):")
print(classification_report(y_clf, y_pred_loo,
                             target_names=["Low Impact", "High Impact"],
                             zero_division=0))

# Fit on full data for coefficients
clf.fit(X_clf, y_clf)

# Figure 19 — confusion matrix + logistic coefficients
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm   = confusion_matrix(y_clf, y_pred_loo)
disp = ConfusionMatrixDisplay(cm, display_labels=["Low Impact", "High Impact"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"M3.2 -- LOO Confusion Matrix\nAcc = {acc:.2f},  AUC = {auc:.2f}")

ax2 = axes[1]
coef_vals  = clf.coef_[0]
feat_order = np.argsort(np.abs(coef_vals))[::-1]
ax2.barh([features[i] for i in feat_order],
         coef_vals[feat_order],
         color=[COLORS["post"] if c > 0 else COLORS["low"]
                for c in coef_vals[feat_order]],
         edgecolor="white")
ax2.set_xlabel("Logistic Coefficient (log-odds direction)")
ax2.set_title("M3.2 -- Classifier Coefficients")
ax2.axvline(0, color="black", lw=0.8)

plt.tight_layout()
save_fig(19, "classifier_confusion_matrix_coefficients")


### Figure 20 — state map: DC count vs post-2022 price (combined)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter_df = model_df.copy()
scatter_df["label"] = scatter_df["stateid"]
ax.scatter(scatter_df["dc_count"], scatter_df["avg_price_post22"],
           c=scatter_df["co2_intensity"], cmap="RdYlGn_r",
           s=100, edgecolors="white", linewidths=0.5, zorder=3)
sm = plt.cm.ScalarMappable(
    cmap="RdYlGn_r",
    norm=plt.Normalize(scatter_df["co2_intensity"].min(),
                       scatter_df["co2_intensity"].max()))
sm.set_array([])
plt.colorbar(sm, ax=ax, label="CO₂ Intensity (×10³ kg/MWh equiv.)")

# label DC-heavy + high-price states
for _, row in scatter_df.iterrows():
    if row["dc_count"] > 20 or row["avg_price_post22"] > 18:
        ax.annotate(row["label"],
                    (row["dc_count"], row["avg_price_post22"]),
                    textcoords="offset points", xytext=(4, 3),
                    fontsize=8)

# regression line
m2, b2 = np.polyfit(scatter_df["dc_count"],
                     scatter_df["avg_price_post22"], 1)
xr = np.linspace(0, scatter_df["dc_count"].max(), 100)
ax.plot(xr, m2 * xr + b2, "--", color="#E63946", lw=1.5,
        label=f"OLS trend  (r={np.corrcoef(scatter_df['dc_count'], scatter_df['avg_price_post22'])[0,1]:.2f})")
ax.set_xlabel("AI Data Center Facility Count")
ax.set_ylabel("Post-2022 Avg. Electricity Price (¢/kWh)")
ax.set_title("Figure 20 — State Electricity Price vs. Data Center Density\n"
             "Color = CO₂ Intensity")
ax.legend(fontsize=9)
save_fig(20, "state_price_vs_dc_density_colored_co2")


# ════════════════════════════════════════════════════════════════════════════


## Model A: Difference-In-Differences

Source: `model_a_did.py`

MODEL A: Difference-in-Differences on Residential-Commercial Price Gap
Research Question:
In states with high AI data center concentration, did the gap between
residential and commercial electricity prices widen faster after 2022
compared to matched low-DC control states?

Hypothesis: β3 > 0 (the DiD coefficient) — the residential-commercial price
gap widened MORE in high-DC states than controls after 2022, implying that
AI data center load is disproportionately shifting costs onto households.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import warnings
warnings.filterwarnings('ignore')


### CONFIGURATION

In [ ]:
SEED = 42
np.random.seed(SEED)

DATA_PATH = "research_data/03_electricity_prices/retail_sales_price_revenue_monthly_all_states.csv"
OUT_DIR   = "results/model_a_did"

# State groups per research design (see claude_code_instructions.txt Section 5)
TREATMENT_STATES = {
    "Virginia":   "VA",
    "Texas":      "TX",
    "California": "CA",
    "Nevada":     "NV",
    "Illinois":   "IL",
}
CONTROL_STATES = {
    "Iowa":          "IA",
    "West Virginia": "WV",
    "Minnesota":     "MN",
    "Tennessee":     "TN",
}

ALL_STATES = {**TREATMENT_STATES, **CONTROL_STATES}

# DiD cutoff: Jan 2022 (AI boom + Russia-Ukraine energy shock)
POST_CUTOFF = "2022-01"


### 1. LOAD AND CLEAN DATA

In [ ]:
print("=" * 70)
print("MODEL A: Difference-in-Differences on Residential-Commercial Price Gap")
print("=" * 70)
print("\n[1] Loading retail sales price data...")

df_raw = pd.read_csv(DATA_PATH)
print(f"    Raw rows: {len(df_raw):,}  |  Columns: {list(df_raw.columns)}")

# Keep only residential and commercial sectors
df = df_raw[df_raw['sectorName'].isin(['residential', 'commercial'])].copy()

# Keep only our 9 focus states by abbreviation
df = df[df['stateid'].isin(ALL_STATES.values())].copy()

# Parse period as date
df['date'] = pd.to_datetime(df['period'])

# Drop rows with missing prices
df = df.dropna(subset=['price'])
print(f"    After filtering: {len(df):,} rows")
print(f"    Date range: {df['date'].min().strftime('%Y-%m')} to {df['date'].max().strftime('%Y-%m')}")
print(f"    States: {sorted(df['stateid'].unique())}")


### 2. CONSTRUCT THE GAP VARIABLE

In [ ]:
print("\n[2] Computing residential-commercial price gap...")

# Pivot to wide: one row per (state, month), with RES and COM as columns
pivot = df.pivot_table(
    index=['stateid', 'stateDescription', 'date'],
    columns='sectorName',
    values='price',
    aggfunc='mean'
).reset_index()

pivot.columns.name = None
pivot = pivot.rename(columns={'residential': 'price_res', 'commercial': 'price_com'})

# GAP: residential minus commercial (cents/kWh)
# Positive gap = households pay more than businesses
pivot['gap'] = pivot['price_res'] - pivot['price_com']

# Ratio: residential / commercial (scale-invariant alternative)
pivot['ratio'] = pivot['price_res'] / pivot['price_com']

# Drop rows where either price is missing
pivot = pivot.dropna(subset=['price_res', 'price_com'])
print(f"    Panel observations: {len(pivot):,}")
print(f"    Average gap by state:")
avg_gap = pivot.groupby('stateid')['gap'].mean().sort_values(ascending=False)
print(avg_gap.to_string())


### 3. CREATE DiD VARIABLES

In [ ]:
print("\n[3] Creating DiD indicator variables...")

# HighDC indicator
pivot['high_dc'] = pivot['stateid'].isin(TREATMENT_STATES.values()).astype(int)

# Post2022 indicator
pivot['post2022'] = (pivot['date'] >= pd.Timestamp(POST_CUTOFF)).astype(int)

# DiD interaction term
pivot['did'] = pivot['high_dc'] * pivot['post2022']

# State and month fixed effects dummies
pivot['state_fe'] = pivot['stateid']
pivot['month_fe'] = pivot['date'].dt.month.astype(str)  # 1-12 seasonal FE
pivot['year_fe']  = pivot['date'].dt.year.astype(str)   # Year FE (absorbs trends)

# Limit to 2015-2025 for cleaner analysis (data exists for all states)
pivot = pivot[pivot['date'].between('2015-01-01', '2025-12-31')].copy()
print(f"    Observations after date filter (2015-2025): {len(pivot):,}")

# Save the panel dataset
panel_path = f"{OUT_DIR}/panel_data.csv"
pivot.to_csv(panel_path, index=False)
print(f"    Panel saved to: {panel_path}")


### 4. PARALLEL TRENDS CHECK

In [ ]:
print("\n[4] Parallel trends check (pre-2022 period)...")

pre = pivot[pivot['post2022'] == 0].copy()
pre_avg = pre.groupby(['date', 'high_dc'])['gap'].mean().reset_index()
pre_avg['group'] = pre_avg['high_dc'].map({1: 'High-DC (Treatment)', 0: 'Control'})

fig, ax = plt.subplots(figsize=(12, 5))
for grp, gdf in pre_avg.groupby('group'):
    color = '#e74c3c' if 'High' in grp else '#2980b9'
    ax.plot(gdf['date'], gdf['gap'], label=grp, color=color, linewidth=2)

ax.axvline(pd.Timestamp('2022-01-01'), color='gray', linestyle='--',
           linewidth=1.5, label='Post-2022 cutoff')
ax.set_title("Parallel Trends Check: Residential–Commercial Price Gap\n"
             "(Pre-2022 Period — Treatment vs Control States)", fontsize=13)
ax.set_xlabel("Date")
ax.set_ylabel("Price Gap (cents/kWh)\nResidential − Commercial")
ax.legend(fontsize=11)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/parallel_trends_plot.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/parallel_trends_plot.png")


### 5. FULL TIME SERIES PLOT

In [ ]:
print("\n[5] Full time series plot (2015-2025)...")

full_avg = pivot.groupby(['date', 'high_dc'])['gap'].mean().reset_index()
full_avg['group'] = full_avg['high_dc'].map({1: 'High-DC States', 0: 'Control States'})

fig, ax = plt.subplots(figsize=(13, 5))
for grp, gdf in full_avg.groupby('group'):
    color = '#e74c3c' if 'High' in grp else '#2980b9'
    ax.plot(gdf['date'], gdf['gap'], label=grp, color=color, linewidth=2.2)

ax.axvline(pd.Timestamp('2022-01-01'), color='gray', linestyle='--',
           linewidth=1.5, label='Jan 2022 (DiD cutoff)')
ax.fill_between([pd.Timestamp('2022-01-01'), pd.Timestamp('2025-12-31')],
                ax.get_ylim()[0] if ax.get_ylim()[0] < 0 else -1,
                100, alpha=0.06, color='red', label='Post-2022 period')
ax.set_title("Residential vs. Commercial Electricity Price Gap\n"
             "High-DC States vs. Control States (2015–2025)", fontsize=13)
ax.set_xlabel("Date")
ax.set_ylabel("Gap (cents/kWh)\nResidential − Commercial")
ax.legend(fontsize=11)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/gap_timeseries_plot.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/gap_timeseries_plot.png")

# Individual state gap plot
fig, axes = plt.subplots(3, 3, figsize=(15, 12), sharex=True)
axes = axes.flatten()
state_list = sorted(ALL_STATES.items(), key=lambda x: x[1])

for i, (state_name, state_id) in enumerate(state_list):
    sdf = pivot[pivot['stateid'] == state_id].sort_values('date')
    is_treat = state_id in TREATMENT_STATES.values()
    color = '#e74c3c' if is_treat else '#2980b9'
    label_tag = ' [DC]' if is_treat else ' [CTRL]'
    axes[i].plot(sdf['date'], sdf['gap'], color=color, linewidth=1.5)
    axes[i].axvline(pd.Timestamp('2022-01-01'), color='gray', linestyle='--', linewidth=1)
    axes[i].set_title(f"{state_name}{label_tag}", fontsize=10)
    axes[i].grid(axis='y', alpha=0.3)
    axes[i].set_ylabel("Gap (¢/kWh)", fontsize=8)

plt.suptitle("Residential−Commercial Price Gap by State (2015–2025)\nRed = High-DC Treatment | Blue = Control",
             fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/individual_state_gaps.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/individual_state_gaps.png")


### 6. DiD REGRESSION MODELS

In [ ]:
print("\n[6] Running DiD regressions...")

# Model 1: Basic DiD (no fixed effects) — baseline
model1 = smf.ols(
    "gap ~ high_dc + post2022 + did",
    data=pivot
).fit(cov_type='cluster', cov_kwds={'groups': pivot['stateid']})

# Model 2: DiD + Month FE (seasonal control)
model2 = smf.ols(
    "gap ~ high_dc + post2022 + did + C(month_fe)",
    data=pivot
).fit(cov_type='cluster', cov_kwds={'groups': pivot['stateid']})

# Model 3: DiD + State FE + Month FE (full specification — primary model)
model3 = smf.ols(
    "gap ~ post2022 + did + C(state_fe) + C(month_fe)",
    data=pivot
).fit(cov_type='cluster', cov_kwds={'groups': pivot['stateid']})

# Model 4: Robustness — exclude 2022 to test if effect persists beyond energy crisis
pivot_excl22 = pivot[~pivot['date'].between('2022-01-01', '2022-12-31')].copy()
model4 = smf.ols(
    "gap ~ post2022 + did + C(state_fe) + C(month_fe)",
    data=pivot_excl22
).fit(cov_type='cluster', cov_kwds={'groups': pivot_excl22['stateid']})


### Print results

In [ ]:
print("\n" + "=" * 70)
print("PRIMARY MODEL (Model 3): DiD + State FE + Month FE")
print("=" * 70)

# Extract key coefficient
did_coef  = model3.params.get('did', np.nan)
did_pval  = model3.pvalues.get('did', np.nan)
did_se    = model3.bse.get('did', np.nan)
did_ci    = model3.conf_int().loc['did'] if 'did' in model3.params.index else [np.nan, np.nan]

print(f"\n  DiD Coefficient (B3):  {did_coef:.4f} cents/kWh")
print(f"  Std. Error:            {did_se:.4f}")
print(f"  p-value:               {did_pval:.4f}")
print(f"  95% CI:                [{did_ci[0]:.4f}, {did_ci[1]:.4f}]")
print(f"  R-squared:             {model3.rsquared:.4f}")
print(f"  N (observations):      {int(model3.nobs):,}")

sig = "***" if did_pval < 0.01 else "**" if did_pval < 0.05 else "*" if did_pval < 0.10 else "(not significant)"
print(f"  Significance:          {sig}")

print("\n  Interpretation:")
if did_pval < 0.10:
    direction = "widened" if did_coef > 0 else "narrowed"
    print(f"  The residential-commercial price gap {direction} by {abs(did_coef):.2f} cents/kWh")
    print(f"  MORE in high-DC states than controls after Jan 2022.")
    print(f"  This supports the hypothesis that AI data center load")
    print(f"  disproportionately shifts costs onto residential consumers.")
else:
    print(f"  The DiD coefficient is not statistically significant at 10% level.")
    print(f"  The price gap change post-2022 is not detectably different between")
    print(f"  high-DC and control states in this sample.")

print("\n" + "=" * 70)
print("ROBUSTNESS CHECK (Model 4): Excluding 2022 (energy crisis year)")
print("=" * 70)
did_coef4 = model4.params.get('did', np.nan)
did_pval4 = model4.pvalues.get('did', np.nan)
did_se4   = model4.bse.get('did', np.nan)
print(f"  DiD (β3):  {did_coef4:.4f}  SE: {did_se4:.4f}  p={did_pval4:.4f}")


### 7. SAVE COEFFICIENT TABLE

In [ ]:
print("\n[7] Saving regression results...")

# Build results table
results = pd.DataFrame({
    'Model': ['Basic DiD', 'DiD + Month FE', 'DiD + State+Month FE (Primary)',
              'Primary excl. 2022 (Robustness)'],
    'DiD_coef_beta3': [
        model1.params.get('did', np.nan),
        model2.params.get('did', np.nan),
        model3.params.get('did', np.nan),
        model4.params.get('did', np.nan),
    ],
    'Std_Error': [
        model1.bse.get('did', np.nan),
        model2.bse.get('did', np.nan),
        model3.bse.get('did', np.nan),
        model4.bse.get('did', np.nan),
    ],
    'p_value': [
        model1.pvalues.get('did', np.nan),
        model2.pvalues.get('did', np.nan),
        model3.pvalues.get('did', np.nan),
        model4.pvalues.get('did', np.nan),
    ],
    'R_squared': [model1.rsquared, model2.rsquared, model3.rsquared, model4.rsquared],
    'N': [int(model1.nobs), int(model2.nobs), int(model3.nobs), int(model4.nobs)],
    'State_FE': ['No', 'No', 'Yes', 'Yes'],
    'Month_FE': ['No', 'Yes', 'Yes', 'Yes'],
    'Excludes_2022': ['No', 'No', 'No', 'Yes'],
})

results.to_csv(f"{OUT_DIR}/did_regression_results.csv", index=False)
print(f"    Saved: {OUT_DIR}/did_regression_results.csv")
print(results.to_string(index=False))


### 8. COEFFICIENT VISUALIZATION

In [ ]:
print("\n[8] Creating coefficient plot...")

model_labels = [
    "Basic DiD\n(no FE)",
    "DiD +\nMonth FE",
    "DiD + State\n+ Month FE\n(Primary)",
    "Primary excl.\n2022\n(Robustness)"
]

coefs  = results['DiD_coef_beta3'].values
ses    = results['Std_Error'].values
pvals  = results['p_value'].values
ci_low = coefs - 1.96 * ses
ci_hi  = coefs + 1.96 * ses

colors = ['#e74c3c' if p < 0.05 else '#f39c12' if p < 0.10 else '#95a5a6'
          for p in pvals]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(model_labels))
ax.bar(x, coefs, color=colors, alpha=0.8, width=0.5)
ax.errorbar(x, coefs, yerr=[coefs - ci_low, ci_hi - coefs],
            fmt='none', color='black', capsize=6, linewidth=2)
ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax.set_xticks(x)
ax.set_xticklabels(model_labels, fontsize=10)
ax.set_ylabel("DiD Coefficient B3 (cents/kWh)", fontsize=11)
ax.set_title("DiD Estimate: How Much MORE Did the Residential–Commercial\n"
             "Price Gap Widen in High-DC States After 2022?", fontsize=12)
ax.annotate("Red = p<0.05  Orange = p<0.10  Gray = not significant",
            xy=(0.5, 0.02), xycoords='axes fraction', ha='center', fontsize=9,
            color='gray')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/did_coefficient_plot.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/did_coefficient_plot.png")


### 9. PRE/POST AVERAGE GAP TABLE

In [ ]:
print("\n[9] Pre/Post average gap by state...")
summary = pivot.groupby(['stateid', 'stateDescription', 'high_dc', 'post2022'])['gap'].mean().reset_index()
summary['period'] = summary['post2022'].map({0: 'Pre-2022', 1: 'Post-2022'})
summary_wide = summary.pivot_table(
    index=['stateid', 'stateDescription', 'high_dc'],
    columns='period', values='gap'
).reset_index()
summary_wide.columns.name = None
summary_wide['Gap_Widening'] = summary_wide['Post-2022'] - summary_wide['Pre-2022']
summary_wide['Group'] = summary_wide['high_dc'].map({1: 'High-DC (Treatment)', 0: 'Control'})
summary_wide = summary_wide.sort_values(['Group', 'Gap_Widening'], ascending=[True, False])
summary_wide.to_csv(f"{OUT_DIR}/prepost_gap_by_state.csv", index=False)
print(summary_wide[['stateid', 'stateDescription', 'Group', 'Pre-2022', 'Post-2022', 'Gap_Widening']].to_string(index=False))
print(f"    Saved: {OUT_DIR}/prepost_gap_by_state.csv")


## Model A Extension: Event Study

Source: `model_a_event_study.py`

MODEL A EXTENSION: Event Study Regression + West Virginia Sensitivity
This script adds two hardening analyses to Model A (DiD):

1. EVENT STUDY REGRESSION
The standard econometric validation for a DiD design. Instead of a
single Post2022 dummy, we estimate year-by-year interaction coefficients
(HighDC × Year_t) relative to a reference year (2021).

If parallel trends hold, all pre-2022 coefficients should be:
- Near zero (no pre-treatment divergence)
- Statistically insignificant

Post-2022 coefficients should be positive and grow over time
(as the AI buildout accelerates and its cost effects compound).

A clean event study plot is the "smoking gun" for DiD credibility.
Any econometrics-literate reviewer will ask for this.

2. WEST VIRGINIA SENSITIVITY
WV widened its residential-commercial gap by +1.18 cents/kWh post-2022
despite being a control state — comparable to some treatment states.
WV explanation: coal plant retirements drove unusual retail restructuring.
This check re-runs the primary DiD excluding WV and formally tests
whether WV is an outlier that biases the main result.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATA_PATH = "research_data/03_electricity_prices/retail_sales_price_revenue_monthly_all_states.csv"
OUT_DIR   = "results/model_a_did"

TREATMENT_STATES = ['VA', 'TX', 'CA', 'NV', 'IL']
CONTROL_STATES   = ['IA', 'WV', 'MN', 'TN']
ALL_STATES       = TREATMENT_STATES + CONTROL_STATES

STATE_NAMES = {
    'VA':'Virginia','TX':'Texas','CA':'California','NV':'Nevada','IL':'Illinois',
    'IA':'Iowa','WV':'W. Virginia','MN':'Minnesota','TN':'Tennessee'
}

print("=" * 70)
print("MODEL A EXTENSION: Event Study + WV Sensitivity")
print("=" * 70)


### LOAD AND PREPARE PANEL

In [ ]:
print("\n[1] Loading and preparing panel...")

df_raw = pd.read_csv(DATA_PATH)
df = df_raw[
    df_raw['sectorName'].isin(['residential', 'commercial']) &
    df_raw['stateid'].isin(ALL_STATES)
].copy()
df['date'] = pd.to_datetime(df['period'])
df = df.dropna(subset=['price'])

pivot = df.pivot_table(
    index=['stateid', 'date'],
    columns='sectorName', values='price', aggfunc='mean'
).reset_index()
pivot.columns.name = None
pivot = pivot.rename(columns={'residential': 'price_res', 'commercial': 'price_com'})
pivot['gap'] = pivot['price_res'] - pivot['price_com']
pivot = pivot.dropna(subset=['gap'])

# Limit to 2015-2025
pivot = pivot[pivot['date'].between('2015-01-01', '2025-12-31')].copy()
pivot['year']    = pivot['date'].dt.year
pivot['month']   = pivot['date'].dt.month.astype(str)
pivot['state_fe']= pivot['stateid']
pivot['high_dc'] = pivot['stateid'].isin(TREATMENT_STATES).astype(int)
pivot['post2022']= (pivot['date'] >= '2022-01-01').astype(int)
pivot['did']     = pivot['high_dc'] * pivot['post2022']

print(f"    Panel: {len(pivot):,} rows | States: {sorted(pivot['stateid'].unique())}")


### EVENT STUDY REGRESSION

In [ ]:
print("\n[2] Event study regression (year-by-year interactions)...")

# Create year dummies (omit 2021 as reference year — last pre-treatment year)
# Interaction: high_dc × year_dummy (relative to 2021)
for yr in range(2015, 2026):
    pivot[f'yr_{yr}'] = (pivot['year'] == yr).astype(int)
    pivot[f'hdc_yr_{yr}'] = pivot['high_dc'] * pivot[f'yr_{yr}']

# Reference year = 2021: omit hdc_yr_2021
event_terms = [f'hdc_yr_{yr}' for yr in range(2015, 2026) if yr != 2021]
event_formula = (
    "gap ~ " +
    " + ".join(event_terms) +
    " + C(state_fe) + C(month)"
)

event_model = smf.ols(event_formula, data=pivot).fit(
    cov_type='cluster', cov_kwds={'groups': pivot['stateid']}
)

# Extract event study coefficients
es_results = []
for yr in range(2015, 2026):
    if yr == 2021:
        # Reference year — coefficient is 0 by construction
        es_results.append({
            'year': yr, 'coef': 0.0, 'se': 0.0, 'pval': np.nan,
            'ci_low': 0.0, 'ci_hi': 0.0, 'is_ref': True
        })
    else:
        term = f'hdc_yr_{yr}'
        coef = event_model.params.get(term, np.nan)
        se   = event_model.bse.get(term, np.nan)
        pval = event_model.pvalues.get(term, np.nan)
        ci   = event_model.conf_int().loc[term] if term in event_model.params.index else [np.nan, np.nan]
        es_results.append({
            'year': yr, 'coef': coef, 'se': se, 'pval': pval,
            'ci_low': ci[0], 'ci_hi': ci[1], 'is_ref': False
        })

es_df = pd.DataFrame(es_results)
es_df.to_csv(f"{OUT_DIR}/event_study_coefficients.csv", index=False)
print("\n    Year-by-year interaction coefficients (reference = 2021):")
print(f"    {'Year':<6} {'Coef':>8} {'SE':>8} {'p-val':>8} {'Sig':>6}")
print("    " + "-" * 42)
for _, row in es_df.iterrows():
    if row['is_ref']:
        print(f"    {int(row['year']):<6} {'[ref]':>8}")
    else:
        sig = "***" if row['pval'] < 0.01 else "**" if row['pval'] < 0.05 else "*" if row['pval'] < 0.10 else ""
        print(f"    {int(row['year']):<6} {row['coef']:>8.4f} {row['se']:>8.4f} {row['pval']:>8.4f} {sig:>6}")


### EVENT STUDY PLOT

In [ ]:
print("\n[3] Event study plot...")

fig, ax = plt.subplots(figsize=(12, 6))

pre_years  = es_df[es_df['year'] < 2022]
post_years = es_df[es_df['year'] >= 2022]

# Pre-treatment: use darker color to highlight pre-trends
ax.errorbar(
    pre_years['year'], pre_years['coef'],
    yerr=[pre_years['coef'] - pre_years['ci_low'],
          pre_years['ci_hi'] - pre_years['coef']],
    fmt='o-', color='#2980b9', linewidth=2, markersize=7,
    capsize=5, label='Pre-treatment years (should be ~0)'
)

# Post-treatment: red
ax.errorbar(
    post_years['year'], post_years['coef'],
    yerr=[post_years['coef'] - post_years['ci_low'],
          post_years['ci_hi'] - post_years['coef']],
    fmt='s-', color='#e74c3c', linewidth=2.5, markersize=8,
    capsize=5, label='Post-treatment years'
)

# Reference year marker
ax.scatter([2021], [0], color='black', s=100, zorder=5, label='Reference year (2021)')

# Vertical line at treatment
ax.axvline(2021.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.fill_between([2021.5, 2025.5], -0.5, 5, alpha=0.06, color='red')

# Significance stars
for _, row in es_df.iterrows():
    if not row['is_ref'] and row['pval'] < 0.10:
        stars = "***" if row['pval'] < 0.001 else "**" if row['pval'] < 0.01 else "*"
        ax.annotate(stars, (row['year'], row['ci_hi'] + 0.12),
                    ha='center', fontsize=9, color='#c0392b')

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Residential–Commercial Price Gap\n(cents/kWh, relative to 2021)", fontsize=11)
ax.set_title(
    "Event Study: Differential Price Gap Between High-DC and Control States\n"
    "(Relative to 2021 Reference Year | State + Month Fixed Effects | Clustered SEs)",
    fontsize=12
)
ax.legend(fontsize=10, loc='upper left')
ax.set_xticks(range(2015, 2026))
ax.grid(axis='y', alpha=0.3)

# Annotate parallel trends region
ax.annotate(
    "Parallel Trends Zone\n(coefficients near 0 →\nassumption satisfied)",
    xy=(2018, -0.3), fontsize=8.5, color='#2980b9',
    bbox=dict(boxstyle='round,pad=0.3', fc='#eaf4fb', ec='#2980b9', alpha=0.8)
)
ax.annotate(
    "Treatment Effect\n(growing divergence →\nDC cost shifting)",
    xy=(2023.2, post_years[post_years['year']==2023]['coef'].values[0] + 0.1),
    fontsize=8.5, color='#c0392b',
    bbox=dict(boxstyle='round,pad=0.3', fc='#fdecea', ec='#e74c3c', alpha=0.8)
)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/event_study_plot.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/event_study_plot.png")

# Validation summary
pre_coefs = es_df[(es_df['year'] < 2021) & (~es_df['is_ref'])]['coef']
post_coefs = es_df[(es_df['year'] > 2021)]['coef']
n_pre_sig  = ((es_df['year'] < 2021) & (~es_df['is_ref']) & (es_df['pval'] < 0.10)).sum()
print(f"\n    Pre-treatment (2015-2020) coefficients:")
print(f"      Mean: {pre_coefs.mean():.4f} | Max abs: {pre_coefs.abs().max():.4f}")
print(f"      Significant at 10%: {n_pre_sig}/{len(pre_coefs)} years")
print(f"    Post-treatment (2022-2025) coefficients:")
print(f"      All positive: {(post_coefs > 0).all()} | Max: {post_coefs.max():.4f}")
if n_pre_sig == 0:
    print("\n    PARALLEL TRENDS: SATISFIED — no pre-treatment divergence detected.")
else:
    print(f"\n    CAUTION: {n_pre_sig} pre-treatment year(s) show significance — inspect plot.")


### WEST VIRGINIA SENSITIVITY

In [ ]:
print("\n[4] West Virginia sensitivity analysis...")

results_table = []

specs = [
    ("Primary (all 9 states)",      pivot,                                None),
    ("Excl. West Virginia",          pivot[pivot['stateid'] != 'WV'],      "WV"),
    ("Excl. 2022 energy crisis yr",  pivot[~pivot['date'].between('2022-01-01','2022-12-31')], None),
    ("Excl. WV + excl. 2022",        pivot[(pivot['stateid'] != 'WV') & (~pivot['date'].between('2022-01-01','2022-12-31'))], "WV"),
]

print(f"\n    {'Specification':<40} {'B3':>8} {'SE':>8} {'p':>8} {'R2':>6}")
print("    " + "-" * 76)

for label, data, _ in specs:
    m = smf.ols(
        "gap ~ post2022 + did + C(state_fe) + C(month)",
        data=data
    ).fit(cov_type='cluster', cov_kwds={'groups': data['stateid']})
    b3 = m.params.get('did', np.nan)
    se = m.bse.get('did', np.nan)
    pv = m.pvalues.get('did', np.nan)
    r2 = m.rsquared
    sig = "***" if pv < 0.01 else "**" if pv < 0.05 else "*" if pv < 0.10 else ""
    print(f"    {label:<40} {b3:>+8.4f} {se:>8.4f} {pv:>8.4f} {r2:>6.4f} {sig}")
    results_table.append({
        'Specification': label,
        'B3_coef': round(b3, 4),
        'Std_Error': round(se, 4),
        'p_value': round(pv, 4),
        'R_squared': round(r2, 4),
        'Significant': sig if sig else 'n.s.',
        'N': int(m.nobs),
    })

sens_df = pd.DataFrame(results_table)
sens_df.to_csv(f"{OUT_DIR}/wv_sensitivity_results.csv", index=False)
print(f"\n    Saved: {OUT_DIR}/wv_sensitivity_results.csv")


### SENSITIVITY VISUALIZATION

In [ ]:
print("\n[5] Sensitivity forest plot...")

labels = [r['Specification'] for r in results_table]
coefs  = [r['B3_coef'] for r in results_table]
ses    = [r['Std_Error'] for r in results_table]
pvals  = [r['p_value'] for r in results_table]
ci_lo  = [c - 1.96*s for c,s in zip(coefs, ses)]
ci_hi  = [c + 1.96*s for c,s in zip(coefs, ses)]

colors = ['#e74c3c' if p < 0.01 else '#f39c12' if p < 0.05 else '#3498db'
          for p in pvals]

fig, ax = plt.subplots(figsize=(10, 4.5))
y = np.arange(len(labels))[::-1]
ax.barh(y, coefs, height=0.35, color=colors, alpha=0.85, label='B3 estimate')
ax.errorbar(coefs, y, xerr=[np.array(coefs)-np.array(ci_lo),
                             np.array(ci_hi)-np.array(coefs)],
            fmt='none', color='black', capsize=6, linewidth=2)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("DiD Coefficient B3 (cents/kWh)", fontsize=11)
ax.set_title(
    "Sensitivity Analysis: DiD Estimate Across Specifications\n"
    "Red=p<0.01  Orange=p<0.05  Blue=p<0.10",
    fontsize=11
)
for i, (c, p) in enumerate(zip(coefs, pvals)):
    star = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""
    ax.text(max(coefs)+0.1, y[i], f"B3={c:+.3f} {star}", va='center', fontsize=9)
ax.set_xlim(left=min(ci_lo)-0.3, right=max(coefs)+0.8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/sensitivity_forest_plot.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/sensitivity_forest_plot.png")


## Synthetic Control

Source: `synthetic_control_virginia.py`

SYNTHETIC CONTROL: Virginia Residential Price Counterfactual
Instead of comparing Virginia to a fixed control group (DiD approach),
synthetic control constructs a "synthetic Virginia" — a weighted combination
of donor states that best replicates Virginia's pre-2022 price trajectory.

After 2022, actual Virginia is compared to its counterfactual (what prices
would have been without the AI data center buildout). The gap is the
estimated causal effect of data center concentration.

Why this complements the DiD:
- DiD uses a fixed control group (IA, WV, MN, TN) with equal weights
- Synthetic control optimally weights ALL donor states to maximize
pre-treatment fit — no researcher degree-of-freedom in control selection
- Visually compelling: one chart shows actual vs. counterfactual
- Preferred in causal inference when n=1 treated unit is the focus

Donor pool: All US states EXCEPT Virginia and other high-DC states
(CA, TX, NV, IL excluded from donor pool as they are treatment states)

Method: pysyncon library (Abadie, Diamond, Hainmueller 2010 implementation)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

try:
    from pysyncon import Dataprep, Synth
    HAS_PYSYNCON = True
except ImportError:
    HAS_PYSYNCON = False
    print("pysyncon not installed — using manual optimization fallback")

from scipy.optimize import minimize

SEED = 42
np.random.seed(SEED)

DATA_PATH = "research_data/03_electricity_prices/retail_sales_price_revenue_monthly_all_states.csv"
OUT_DIR   = "results/model_synthetic_control"

# Exclude treatment states from donor pool
TREATMENT_STATES = ['VA', 'TX', 'CA', 'NV', 'IL']
EXCLUDE_STATES   = TREATMENT_STATES + ['HI', 'AK', 'DC']  # also exclude outlier states

# Use annual averages (synthetic control works best with annual data)
TREATMENT_UNIT   = 'VA'
TREATMENT_NAME   = 'Virginia'
TREATMENT_YEAR   = 2022   # intervention year

print("=" * 70)
print("SYNTHETIC CONTROL: Virginia Residential Price Counterfactual")
print("=" * 70)


### 1. LOAD AND PREPARE DATA

In [ ]:
print("\n[1] Loading price data...")

df_raw = pd.read_csv(DATA_PATH)
df_raw['date'] = pd.to_datetime(df_raw['period'])

# Keep residential prices for US states only
us_states = [
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA',
    'KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT',
    'VA','WA','WV','WI','WY'
]

df = df_raw[
    (df_raw['stateid'].isin(us_states)) &
    (df_raw['sectorName'] == 'residential') &
    (df_raw['date'].between('2015-01-01', '2025-12-31'))
].copy()
df = df.dropna(subset=['price'])

# Annual averages
df['year'] = df['date'].dt.year
annual = df.groupby(['stateid', 'year'])['price'].mean().reset_index()
annual.columns = ['state', 'year', 'res_price']

# Pivot to wide: rows=year, columns=state
wide = annual.pivot(index='year', columns='state', values='res_price')
wide = wide.dropna(axis=1, thresh=8)  # require 8+ years of data per state

# Donor states
donors = [s for s in wide.columns if s not in EXCLUDE_STATES]
print(f"    Treated: {TREATMENT_UNIT} | Donors: {len(donors)} states")
print(f"    Years: {wide.index.min()} - {wide.index.max()}")
print(f"    Donor states: {donors}")

# Pre-treatment and post-treatment periods
pre_years  = [y for y in wide.index if y < TREATMENT_YEAR]
post_years = [y for y in wide.index if y >= TREATMENT_YEAR]
print(f"    Pre-treatment: {pre_years} | Post-treatment: {post_years}")


### 2. SYNTHETIC CONTROL OPTIMIZATION

In [ ]:
print("\n[2] Optimizing synthetic control weights...")

# Outcome matrix: pre-treatment only
Y_pre = wide.loc[pre_years, donors].values   # shape: (n_pre, n_donors)
y_va_pre = wide.loc[pre_years, TREATMENT_UNIT].values  # treated pre-period

Y_all = wide.loc[:, donors].values           # all years
y_va_all = wide.loc[:, TREATMENT_UNIT].values

# Optimize donor weights W to minimize ||y_va_pre - Y_pre @ W||^2
# Subject to: W >= 0, sum(W) = 1

def objective(W):
    synth = Y_pre @ W
    return np.sum((y_va_pre - synth)**2)

def gradient(W):
    synth = Y_pre @ W
    return -2 * Y_pre.T @ (y_va_pre - synth)

n_donors = len(donors)
W0       = np.ones(n_donors) / n_donors

result = minimize(
    objective, W0,
    method='SLSQP',
    jac=gradient,
    constraints=[{'type': 'eq', 'fun': lambda W: np.sum(W) - 1}],
    bounds=[(0, 1)] * n_donors,
    options={'maxiter': 2000, 'ftol': 1e-10}
)

W_opt = result.x
synth_pre  = Y_pre @ W_opt
synth_all  = Y_all @ W_opt

# Pre-treatment RMSPE
pre_rmspe = np.sqrt(np.mean((y_va_pre - synth_pre)**2))
print(f"    Optimization status: {result.message}")
print(f"    Pre-treatment RMSPE: {pre_rmspe:.4f} cents/kWh")

# Top donor states
donor_weights = pd.DataFrame({
    'state': donors, 'weight': W_opt
}).sort_values('weight', ascending=False)
top_donors = donor_weights[donor_weights['weight'] > 0.01]
print(f"\n    Top donor states (weight > 1%):")
print(top_donors.to_string(index=False))
donor_weights.to_csv(f"{OUT_DIR}/synthetic_control_weights.csv", index=False)


### 3. SYNTHETIC CONTROL PLOT

In [ ]:
print("\n[3] Generating synthetic control plot...")

years      = wide.index.tolist()
y_actual   = y_va_all
y_synth    = synth_all

# Treatment effect: actual - synthetic
effect = y_actual - y_synth
post_effect = effect[[i for i, y in enumerate(years) if y >= TREATMENT_YEAR]]
avg_effect  = post_effect.mean()
max_effect  = post_effect.max()

print(f"    Average post-{TREATMENT_YEAR} treatment effect: {avg_effect:+.4f} cents/kWh")
print(f"    Maximum treatment effect:                      {max_effect:+.4f} cents/kWh")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Top: Actual vs Synthetic Virginia
for y_pre in pre_years:
    pass  # just to establish range
pre_mask  = [y < TREATMENT_YEAR for y in years]
post_mask = [y >= TREATMENT_YEAR for y in years]

ax1.plot(years, y_actual, 'o-', color='#e74c3c', linewidth=2.5, markersize=7,
         label=f'Actual {TREATMENT_NAME}')
ax1.plot(years, y_synth, 's--', color='#2980b9', linewidth=2.5, markersize=7,
         label=f'Synthetic {TREATMENT_NAME} (counterfactual)')

# Shade the treatment effect
years_arr = np.array(years)
ax1.fill_between(years_arr, y_actual, y_synth,
                 where=(years_arr >= TREATMENT_YEAR),
                 alpha=0.25, color='#e74c3c',
                 label=f'Treatment effect (avg={avg_effect:+.2f} ¢/kWh)')

ax1.axvline(TREATMENT_YEAR - 0.5, color='gray', linestyle='--', linewidth=1.5)
ax1.axvline(TREATMENT_YEAR - 0.5, color='gray', linestyle='--', linewidth=1.5,
            label='2022 cutoff (AI buildout begins)')

ax1.set_ylabel("Residential Electricity Price (cents/kWh)", fontsize=11)
ax1.set_title(
    f"Synthetic Control: Actual Virginia vs. Counterfactual\n"
    f"(What would residential prices have been without AI data centers?)",
    fontsize=12
)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)
ax1.set_xticks(years)
ax1.set_xticklabels([str(y) for y in years], rotation=0)

# Annotate pre-treatment fit
pre_fit_text = f"Pre-{TREATMENT_YEAR} fit:\nRMSPE = {pre_rmspe:.3f} ¢/kWh"
ax1.annotate(pre_fit_text, xy=(2018, min(y_synth[:len(pre_years)]) - 0.5),
             fontsize=9, color='#2980b9',
             bbox=dict(boxstyle='round', fc='#eaf4fb', ec='#2980b9', alpha=0.8))

# Bottom: Treatment effect (gap)
ax2.bar(years, effect, color=['#e74c3c' if e > 0 else '#2980b9' for e in effect],
        alpha=0.8, width=0.6)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.axvline(TREATMENT_YEAR - 0.5, color='gray', linestyle='--', linewidth=1.5)
ax2.fill_between([TREATMENT_YEAR - 0.5, max(years) + 0.5], -0.1, max(effect) * 1.3,
                 alpha=0.06, color='red')

for i, (yr, eff) in enumerate(zip(years, effect)):
    if yr >= TREATMENT_YEAR:
        ax2.text(yr, eff + 0.04, f'+{eff:.2f}', ha='center', fontsize=9, color='#c0392b')
    else:
        offset = -0.08 if eff < 0 else 0.04
        ax2.text(yr, eff + offset, f'{eff:+.2f}', ha='center', fontsize=8, color='gray')

ax2.set_ylabel("Treatment Effect (Actual - Synthetic)\ncents/kWh", fontsize=11)
ax2.set_xlabel("Year")
ax2.set_title(
    f"Estimated Price Premium Attributable to AI Data Center Concentration\n"
    f"(Positive = Virginia residential prices ABOVE counterfactual)",
    fontsize=11
)
ax2.grid(axis='y', alpha=0.3)
ax2.set_xticks(years)
ax2.set_xticklabels([str(y) for y in years], rotation=0)

plt.suptitle(
    f"SYNTHETIC CONTROL: Virginia Residential Electricity Price\n"
    f"AI Data Center Premium: Avg +{avg_effect:.2f} cents/kWh post-2022",
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/synthetic_control_virginia.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/synthetic_control_virginia.png")


### 4. DONOR WEIGHT VISUALIZATION

In [ ]:
print("\n[4] Donor weight visualization...")

top10 = donor_weights.head(12)
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c' if w > 0.1 else '#f39c12' if w > 0.05 else '#bdc3c7'
          for w in top10['weight']]
ax.barh(top10['state'][::-1], top10['weight'][::-1]*100, color=colors[::-1], alpha=0.85)
ax.set_xlabel("Donor Weight (%)")
ax.set_title("Synthetic Virginia: Donor State Composition\n"
             "(States whose price trajectory best matches Virginia pre-2022)")
ax.grid(axis='x', alpha=0.3)
for i, (_, row) in enumerate(top10[::-1].iterrows()):
    ax.text(row['weight']*100 + 0.3, i, f"{row['weight']*100:.1f}%", va='center', fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/donor_weights.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/donor_weights.png")


### 5. SAVE EFFECT ESTIMATES

In [ ]:
effects_df = pd.DataFrame({
    'year': years,
    'actual_va': y_actual,
    'synthetic_va': y_synth,
    'treatment_effect': effect,
    'is_post_treatment': [y >= TREATMENT_YEAR for y in years],
})
effects_df.to_csv(f"{OUT_DIR}/synthetic_control_effects.csv", index=False)
print(f"    Saved: {OUT_DIR}/synthetic_control_effects.csv")


## Model B: Load Shape Fingerprinting

Source: `model_b_load_shape.py`

MODEL B: ISO-Level Load Shape Comparison — DC-Heavy vs Non-DC Regions
Research Question:
Do power grid regions (ISOs) covering high AI data center concentration
exhibit measurably different hourly load profiles — specifically flatter
daily curves, higher base load, and lower peak-to-trough variability —
compared to non-DC-concentrated ISOs?

Design Note:
The pjm_load_hourly.csv file contains only PJM system-level totals (not
subzone data). We therefore use the multi-BA hourly demand file from
folder 05 (2019-2024) to compare load shape across ISOs.

DC-Heavy ISOs:
PJM  — covers Northern Virginia's Data Center Alley (largest global DC hub)
CISO — covers California's data center clusters (Silicon Valley, LA)

Non-DC ISOs (structural controls):
ISNE — ISO New England (Boston/New England, minimal DC presence)
MISO — Midcontinent ISO (agricultural/industrial midwest)
SWPP — Southwest Power Pool (Oklahoma/Kansas, minimal DC)

AI data centers run 24/7 at ~100% utilization, which should produce a
visible "fingerprint" in load data: flatter daily curves, higher minimum
load, lower peak-to-trough ratio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATA_PATH = "research_data/05_hourly_demand/hourly_demand_major_balancing_authorities.csv"
OUT_DIR   = "results/model_b_load_shape"

# ISO classifications
DC_HEAVY = ['PJM', 'CISO']    # high data center concentration
NON_DC   = ['ISNE', 'MISO', 'SWPP']  # minimal DC presence

# Map respondent codes to names
ISO_NAMES = {
    'PJM':  'PJM (Virginia/DC)',
    'CISO': 'CAISO (California)',
    'ISNE': 'ISO-NE (New England)',
    'MISO': 'MISO (Midwest)',
    'SWPP': 'SW Power Pool',
}


### 1. LOAD DATA

In [ ]:
print("=" * 70)
print("MODEL B: ISO Load Shape Comparison — DC-Heavy vs Non-DC Regions")
print("=" * 70)
print("\n[1] Loading hourly demand data...")

df = pd.read_csv(DATA_PATH)
print(f"    Rows: {len(df):,}  |  ISOs: {sorted(df['respondent'].unique())}")

# Parse datetime
df['dt'] = pd.to_datetime(df['period'].str.replace('T', ' '), errors='coerce')
df = df.dropna(subset=['dt', 'demand_mw'])
df = df[df['demand_mw'] > 0]  # remove zeros/negatives

# Keep only demand rows (not forecast)
df = df[df['type'] == 'D'].copy()

df['date']  = df['dt'].dt.date
df['hour']  = df['dt'].dt.hour
df['month'] = df['dt'].dt.month
df['year']  = df['dt'].dt.year
df['iso']   = df['respondent']

# Remove outliers: cap each ISO at its 99.9th percentile
# (PJM has documented data corruption events, e.g. 2.1 billion MWh on 2021-10-19)
p999 = df.groupby('iso')['demand_mw'].transform(lambda x: x.quantile(0.999))
n_outliers = (df['demand_mw'] > p999).sum()
df = df[df['demand_mw'] <= p999].copy()
print(f"    Removed {n_outliers} outlier rows (> 99.9th pct per ISO)")

print(f"    Date range: {df['dt'].min()} to {df['dt'].max()}")
print(f"    ISOs available: {sorted(df['iso'].unique())}")


### 2. COMPUTE DAILY LOAD SHAPE METRICS

In [ ]:
print("\n[2] Computing daily load shape metrics per ISO...")

def compute_daily_metrics(group):
    """Compute load shape metrics for a single ISO-day."""
    vals = group['demand_mw'].values
    if len(vals) < 12:  # need at least 12 hours for valid metrics
        return None
    daily_mean = vals.mean()
    daily_max  = vals.max()
    daily_min  = vals.min()
    daily_std  = vals.std()

    # Load factor: higher = flatter curve (DC fingerprint: closer to 1.0)
    load_factor = daily_mean / daily_max if daily_max > 0 else np.nan

    # Peak-to-trough: lower = flatter (DC fingerprint: closer to 1.0)
    peak_to_trough = daily_max / daily_min if daily_min > 0 else np.nan

    # Coefficient of variation: lower = more uniform
    cv = daily_std / daily_mean if daily_mean > 0 else np.nan

    # Ramp rate volatility: std of hour-over-hour changes
    hourly_changes = np.diff(vals)
    ramp_volatility = hourly_changes.std() if len(hourly_changes) > 0 else np.nan

    # Base load ratio: minimum as % of maximum
    base_load_ratio = daily_min / daily_max if daily_max > 0 else np.nan

    return pd.Series({
        'load_factor':    load_factor,
        'peak_to_trough': peak_to_trough,
        'cv':             cv,
        'ramp_volatility': ramp_volatility,
        'base_load_ratio': base_load_ratio,
        'daily_mean_mw':  daily_mean,
        'daily_max_mw':   daily_max,
        'daily_min_mw':   daily_min,
        'n_hours':        len(vals),
    })

daily_metrics = (
    df.groupby(['iso', 'date'])
    .apply(compute_daily_metrics, include_groups=False)
    .reset_index()
    .dropna()
)
daily_metrics['date']    = pd.to_datetime(daily_metrics['date'])
daily_metrics['year']    = daily_metrics['date'].dt.year
daily_metrics['month']   = daily_metrics['date'].dt.month
daily_metrics['dc_heavy'] = daily_metrics['iso'].isin(DC_HEAVY).astype(int)
daily_metrics['iso_name'] = daily_metrics['iso'].map(ISO_NAMES).fillna(daily_metrics['iso'])
daily_metrics['post2022'] = (daily_metrics['date'] >= '2022-01-01').astype(int)

# Filter to ISOs of interest
daily_metrics = daily_metrics[daily_metrics['iso'].isin(DC_HEAVY + NON_DC)].copy()

print(f"    Daily metrics computed: {len(daily_metrics):,} rows")
print(f"    ISOs: {sorted(daily_metrics['iso'].unique())}")

summary = daily_metrics.groupby('iso')[['load_factor', 'peak_to_trough', 'cv', 'ramp_volatility', 'base_load_ratio']].mean()
print("\n    Mean daily metrics by ISO:")
print(summary.round(4).to_string())


### 3. STATISTICAL TESTS

In [ ]:
print("\n[3] Statistical tests: DC-Heavy vs Non-DC ISOs...")

dc_data  = daily_metrics[daily_metrics['dc_heavy'] == 1]
non_data = daily_metrics[daily_metrics['dc_heavy'] == 0]

metrics = ['load_factor', 'peak_to_trough', 'cv', 'ramp_volatility', 'base_load_ratio']
metric_labels = {
    'load_factor':    'Load Factor (mean/peak)',
    'peak_to_trough': 'Peak-to-Trough Ratio',
    'cv':             'Coefficient of Variation',
    'ramp_volatility':'Ramp Rate Volatility (MW/hr)',
    'base_load_ratio':'Base Load Ratio (min/max)',
}

test_results = []
for m in metrics:
    dc_vals  = dc_data[m].dropna().values
    non_vals = non_data[m].dropna().values

    # Welch t-test (unequal variances)
    t_stat, t_pval = stats.ttest_ind(dc_vals, non_vals, equal_var=False)

    # Mann-Whitney U (non-parametric)
    u_stat, u_pval = stats.mannwhitneyu(dc_vals, non_vals, alternative='two-sided')

    result = {
        'Metric':       metric_labels[m],
        'DC_Heavy_Mean': dc_vals.mean(),
        'NonDC_Mean':   non_vals.mean(),
        'Difference':   dc_vals.mean() - non_vals.mean(),
        'Welch_t':      t_stat,
        'Welch_p':      t_pval,
        'MannWhitney_p': u_pval,
        'Significant_5pct': 'Yes' if min(t_pval, u_pval) < 0.05 else 'No',
    }
    test_results.append(result)

    direction = "HIGHER" if dc_vals.mean() > non_vals.mean() else "LOWER"
    sig = "***" if min(t_pval, u_pval) < 0.001 else "**" if min(t_pval, u_pval) < 0.01 else "*" if min(t_pval, u_pval) < 0.05 else "n.s."
    print(f"  {metric_labels[m]}: DC={dc_vals.mean():.4f}, Non-DC={non_vals.mean():.4f} | t-p={t_pval:.4f} | MW-p={u_pval:.4f} {sig}")

test_df = pd.DataFrame(test_results)
test_df.to_csv(f"{OUT_DIR}/statistical_tests.csv", index=False)
print(f"\n    Saved: {OUT_DIR}/statistical_tests.csv")


### 4. AVERAGE DAILY LOAD PROFILE PLOTS

In [ ]:
print("\n[4] Plotting average daily load profiles...")

# Normalize demand to % of daily max so ISOs (different sizes) are comparable
df['demand_norm'] = df.groupby(['iso', 'date'])['demand_mw'].transform(
    lambda x: x / x.max() * 100
)

hourly_avg = df[df['iso'].isin(DC_HEAVY + NON_DC)].groupby(['iso', 'hour'])['demand_norm'].mean().reset_index()
hourly_avg['iso_name'] = hourly_avg['iso'].map(ISO_NAMES).fillna(hourly_avg['iso'])
hourly_avg['dc_type'] = hourly_avg['iso'].apply(
    lambda x: 'High-DC ISO' if x in DC_HEAVY else 'Non-DC ISO'
)

colors_dc  = {'PJM': '#e74c3c', 'CISO': '#c0392b'}
colors_non = {'ISNE': '#2980b9', 'MISO': '#1a5276', 'SWPP': '#5dade2'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: all ISOs
for iso, group in hourly_avg.groupby('iso'):
    is_dc = iso in DC_HEAVY
    color = colors_dc.get(iso, colors_non.get(iso, '#999'))
    style = '-' if is_dc else '--'
    lw    = 2.5 if is_dc else 1.8
    ax1.plot(group['hour'], group['demand_norm'],
             label=ISO_NAMES.get(iso, iso), color=color, linestyle=style, linewidth=lw)

ax1.set_xlabel("Hour of Day (0-23)")
ax1.set_ylabel("Demand (% of Daily Peak)")
ax1.set_title("Average Daily Load Profile by ISO\n(Normalized to % of Daily Peak)")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_xticks(range(0, 24, 3))

# Right: group averages
group_avg = hourly_avg.groupby(['hour', 'dc_type'])['demand_norm'].mean().reset_index()
for dtype, group in group_avg.groupby('dc_type'):
    color = '#e74c3c' if 'High' in dtype else '#2980b9'
    ax2.plot(group['hour'], group['demand_norm'], label=dtype, color=color, linewidth=3)

ax2.set_xlabel("Hour of Day (0-23)")
ax2.set_ylabel("Demand (% of Daily Peak)")
ax2.set_title("High-DC vs Non-DC ISOs\nAverage Daily Load Shape")
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)
ax2.set_xticks(range(0, 24, 3))
# Annotate the flattening
lf_dc  = daily_metrics[daily_metrics['dc_heavy'] == 1]['load_factor'].mean()
lf_non = daily_metrics[daily_metrics['dc_heavy'] == 0]['load_factor'].mean()
ax2.annotate(f"High-DC Load Factor: {lf_dc:.3f}", xy=(12, 85), fontsize=9, color='#e74c3c')
ax2.annotate(f"Non-DC Load Factor:  {lf_non:.3f}", xy=(12, 80), fontsize=9, color='#2980b9')

plt.suptitle("DATA CENTER GRID FINGERPRINT: Flatter Daily Load Curves in DC-Heavy ISOs",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/load_profile_comparison.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/load_profile_comparison.png")


### 5. LOAD FACTOR OVER TIME

In [ ]:
print("\n[5] Load factor time series by ISO...")

monthly_lf = (
    daily_metrics
    .groupby(['iso', 'iso_name', pd.Grouper(key='date', freq='ME')])['load_factor']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 5))
for iso in DC_HEAVY + NON_DC:
    if iso not in monthly_lf['iso'].values:
        continue
    sub = monthly_lf[monthly_lf['iso'] == iso]
    is_dc = iso in DC_HEAVY
    color = colors_dc.get(iso, colors_non.get(iso, '#999'))
    style = '-' if is_dc else '--'
    lw    = 2.5 if is_dc else 1.8
    ax.plot(sub['date'], sub['load_factor'], label=ISO_NAMES.get(iso, iso),
            color=color, linestyle=style, linewidth=lw)

ax.axvline(pd.Timestamp('2022-01-01'), color='gray', linestyle=':', linewidth=1.5,
           label='Jan 2022')
ax.set_title("Monthly Average Load Factor by ISO (2019-2024)\nHigher = Flatter Daily Curve = More Baseload Demand")
ax.set_xlabel("Date")
ax.set_ylabel("Load Factor (Daily Mean / Daily Peak)")
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/load_factor_timeseries.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/load_factor_timeseries.png")


### 6. HEATMAP OF LOAD FACTOR BY ISO AND YEAR

In [ ]:
print("\n[6] Heatmap of load factor by ISO and year...")

hm_data = daily_metrics.groupby(['iso', 'year'])['load_factor'].mean().reset_index()
hm_pivot = hm_data.pivot(index='iso', columns='year', values='load_factor')

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(hm_pivot, annot=True, fmt='.3f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Load Factor'})
ax.set_title("Load Factor by ISO and Year\n(Higher = Flatter Curve = More 24/7 Baseload Demand)")
ax.set_xlabel("Year")
ax.set_ylabel("ISO")
# Annotate DC-heavy
for idx, iso in enumerate(hm_pivot.index):
    if iso in DC_HEAVY:
        ax.annotate("DC-Heavy", xy=(-0.1, idx + 0.5), xycoords='data',
                    xytext=(-2.5, idx + 0.5), textcoords='data',
                    fontsize=8, color='#e74c3c',
                    arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1))
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/load_factor_heatmap.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/load_factor_heatmap.png")


### 7. BOX PLOTS OF METRIC DISTRIBUTIONS

In [ ]:
print("\n[7] Box plots of metric distributions...")

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
plot_metrics = [
    ('load_factor',    'Load Factor\n(Higher = Flatter)'),
    ('peak_to_trough', 'Peak-to-Trough Ratio\n(Lower = Flatter)'),
    ('cv',             'Coefficient of Variation\n(Lower = More Uniform)'),
]

daily_metrics['Group'] = daily_metrics['dc_heavy'].map(
    {1: 'High-DC\n(PJM, CAISO)', 0: 'Non-DC\n(ISNE, MISO, SWPP)'}
)

palette = {'High-DC\n(PJM, CAISO)': '#e74c3c', 'Non-DC\n(ISNE, MISO, SWPP)': '#2980b9'}

for i, (metric, ylabel) in enumerate(plot_metrics):
    sns.boxplot(data=daily_metrics, x='Group', y=metric,
                palette=palette, ax=axes[i], fliersize=2)
    axes[i].set_ylabel(ylabel)
    axes[i].set_xlabel("")
    axes[i].grid(axis='y', alpha=0.3)

    # Add p-value annotation
    t = [r for r in test_results if r['Metric'] == metric_labels.get(metric, metric)]
    if t:
        p = min(t[0]['Welch_p'], t[0]['MannWhitney_p'])
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
        axes[i].set_title(f"{stars} (p={p:.4f})")

plt.suptitle("Load Shape Metrics: High-DC ISOs vs Non-DC ISOs", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/metric_boxplots.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/metric_boxplots.png")


### 8. PRE/POST 2022 ANALYSIS

In [ ]:
print("\n[8] Pre/Post 2022 comparison by ISO...")

prepost = daily_metrics.groupby(['iso', 'post2022'])[metrics].mean().reset_index()
prepost['Period'] = prepost['post2022'].map({0: 'Pre-2022', 1: 'Post-2022'})
prepost['iso_name'] = prepost['iso'].map(ISO_NAMES).fillna(prepost['iso'])
prepost['dc_heavy'] = prepost['iso'].isin(DC_HEAVY)

# DiD on load_factor: did DC-heavy ISOs become flatter post-2022?
print("\n  Load Factor DiD (did DC-heavy ISOs get flatter after 2022?):")
did_lf = smf.ols(
    "load_factor ~ dc_heavy + post2022 + dc_heavy:post2022",
    data=daily_metrics
).fit(cov_type='cluster', cov_kwds={'groups': daily_metrics['iso']})

did_coef  = did_lf.params.get('dc_heavy:post2022', np.nan)
did_pval  = did_lf.pvalues.get('dc_heavy:post2022', np.nan)
did_se    = did_lf.bse.get('dc_heavy:post2022', np.nan)
print(f"  DiD coefficient: {did_coef:.5f}  SE: {did_se:.5f}  p={did_pval:.4f}")

# Per-ISO pre/post load factor
print("\n  Per-ISO load factor pre/post 2022:")
pp = prepost[['iso_name', 'Period', 'load_factor']].pivot(
    index='iso_name', columns='Period', values='load_factor'
).reset_index()
pp['Change'] = pp['Post-2022'] - pp['Pre-2022']
print(pp.to_string(index=False))
pp.to_csv(f"{OUT_DIR}/prepost_load_factor.csv", index=False)
prepost.to_csv(f"{OUT_DIR}/prepost_all_metrics.csv", index=False)
print(f"    Saved: {OUT_DIR}/prepost_load_factor.csv")


## Model C: ERCOT Vulnerability Classifier

Source: `model_c_ercot.py`

MODEL C: ERCOT Grid Vulnerability Classifier
Research Question:
In ERCOT's isolated Texas grid, can we predict hours of critical grid
stress (high load conditions) using load data, renewable generation share,
time features, and data center capacity?

Why ERCOT:
ERCOT is isolated — no interstate power imports — making it a clean,
closed system for modeling. Texas has 877 data center facilities
totaling ~67,254 MW installed capacity (Aterio atlas). High load hours
that strain the grid disproportionately affect reliability and price.

Target Variable:
Binary: 1 = "stressed" hour (load >= 90th percentile for that month)
0 = "normal operations"
Rationale: Monthly percentile controls for seasonal variation; the 90th
percentile captures genuinely high-demand events without being too rare.

Features:
- Lagged load: 1h, 6h, 12h, 24h, 48h
- Rolling statistics: 24h mean, 24h std, 7-day mean
- Renewable share: (wind + solar) / total_generation
- Fossil share: (coal + natural gas) / total_generation
- Time features: hour, day_of_week, month, year
- Seasonal indicators: summer, winter
- Texas DC capacity (MW) — used as a static feature for 2023 vs 2024

Model: XGBoost binary classifier with time-series cross-validation.
IMPORTANT: uses walk-forward (expanding window) CV, not random CV.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, average_precision_score,
    f1_score
)
from sklearn.calibration import calibration_curve
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

LOAD_PATH     = "research_data/07_grid_stress/erco_load_hourly.csv"
FUEL_PATH     = "research_data/07_grid_stress/erco_fuel_mix_hourly.csv"
FORECAST_PATH = "research_data/07_grid_stress/erco_load_forecast_hourly.csv"
OUT_DIR       = "results/model_c_ercot"

# Texas DC capacity by year (from Aterio atlas total: 67,254 MW)
# Approximate ramp-up: 2023 base, 2024 expanded
TX_DC_CAPACITY = {2023: 45000, 2024: 67254}


### 1. LOAD AND MERGE DATA

In [ ]:
print("=" * 70)
print("MODEL C: ERCOT Grid Vulnerability Classifier")
print("=" * 70)
print("\n[1] Loading ERCOT load and fuel mix data...")

# --- Load ---
df_load = pd.read_csv(LOAD_PATH)
df_load = df_load[df_load['type'] == 'D'].copy()
df_load['dt'] = pd.to_datetime(df_load['period'].str.replace('T', ' '), errors='coerce')
df_load = df_load.dropna(subset=['dt', 'demand_mw'])
df_load = df_load.rename(columns={'demand_mw': 'load_mw'})
df_load = df_load[['dt', 'load_mw']].set_index('dt').sort_index()
print(f"    Load rows: {len(df_load):,}  |  Range: {df_load.index.min()} to {df_load.index.max()}")

# --- Fuel Mix ---
df_fuel = pd.read_csv(FUEL_PATH)
df_fuel['dt'] = pd.to_datetime(df_fuel['period'].str.replace('T', ' '), errors='coerce')
df_fuel = df_fuel.dropna(subset=['dt', 'generation_mw'])

# Pivot to wide: one column per fuel type
fuel_wide = df_fuel.pivot_table(index='dt', columns='fueltype', values='generation_mw', aggfunc='sum')
fuel_wide.columns = [f'gen_{c.lower()}' for c in fuel_wide.columns]
fuel_wide = fuel_wide.fillna(0)

# Compute renewable and fossil shares
# Renewables: wind (WND), solar (SUN), hydro (WAT)
# Fossils: coal (COL), natural gas (NG)
fuel_wide['total_gen'] = fuel_wide.sum(axis=1)
fuel_wide['renewable_gen'] = fuel_wide.get('gen_wnd', 0) + fuel_wide.get('gen_sun', 0) + fuel_wide.get('gen_wat', 0)
fuel_wide['fossil_gen']    = fuel_wide.get('gen_col', 0) + fuel_wide.get('gen_ng', 0)
fuel_wide['wind_gen']      = fuel_wide.get('gen_wnd', 0)
fuel_wide['solar_gen']     = fuel_wide.get('gen_sun', 0)

fuel_wide['renewable_share'] = fuel_wide['renewable_gen'] / fuel_wide['total_gen'].replace(0, np.nan)
fuel_wide['fossil_share']    = fuel_wide['fossil_gen'] / fuel_wide['total_gen'].replace(0, np.nan)
fuel_wide['wind_share']      = fuel_wide['wind_gen'] / fuel_wide['total_gen'].replace(0, np.nan)
fuel_wide['solar_share']     = fuel_wide['solar_gen'] / fuel_wide['total_gen'].replace(0, np.nan)

print(f"    Fuel mix rows: {len(fuel_wide):,}")

# --- Forecast (load forecast for error feature) ---
try:
    df_fc = pd.read_csv(FORECAST_PATH, nrows=10)
    print(f"    Forecast columns: {df_fc.columns.tolist()}")
    has_forecast = True
except:
    has_forecast = False
    print("    No forecast file available")

# --- Merge ---
df = df_load.join(fuel_wide[['renewable_share', 'fossil_share', 'wind_share',
                               'solar_share', 'total_gen', 'renewable_gen']], how='left')
df = df.dropna(subset=['load_mw'])
print(f"    Merged dataset: {len(df):,} rows")


### 2. TARGET VARIABLE DEFINITION

In [ ]:
print("\n[2] Defining stress target variable...")

# Monthly 90th percentile threshold per year-month
df['year_month'] = df.index.to_period('M')
df['monthly_p90'] = df.groupby('year_month')['load_mw'].transform(
    lambda x: x.quantile(0.90)
)
df['stressed'] = (df['load_mw'] >= df['monthly_p90']).astype(int)

n_stressed = df['stressed'].sum()
n_total    = len(df)
print(f"    Stressed hours: {n_stressed:,} ({n_stressed/n_total*100:.1f}%)")
print(f"    Normal hours:   {n_total - n_stressed:,} ({(n_total-n_stressed)/n_total*100:.1f}%)")
print(f"    (Target: ~10% stressed, class imbalance handled via scale_pos_weight)")


### 3. FEATURE ENGINEERING

In [ ]:
print("\n[3] Engineering features...")

df = df.reset_index()
df = df.rename(columns={'index': 'dt'})
df = df.sort_values('dt').reset_index(drop=True)

# Time features
df['hour']       = df['dt'].dt.hour
df['dow']        = df['dt'].dt.dayofweek  # 0=Monday, 6=Sunday
df['month']      = df['dt'].dt.month
df['year']       = df['dt'].dt.year
df['is_weekend'] = (df['dow'] >= 5).astype(int)
df['is_summer']  = df['month'].isin([6, 7, 8]).astype(int)
df['is_winter']  = df['month'].isin([12, 1, 2]).astype(int)
df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin']  = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']  = np.cos(2 * np.pi * df['month'] / 12)

# Lagged load features
for lag in [1, 6, 12, 24, 48]:
    df[f'load_lag_{lag}h'] = df['load_mw'].shift(lag)

# Rolling statistics
df['load_roll_24h_mean'] = df['load_mw'].rolling(24, min_periods=12).mean().shift(1)
df['load_roll_24h_std']  = df['load_mw'].rolling(24, min_periods=12).std().shift(1)
df['load_roll_7d_mean']  = df['load_mw'].rolling(168, min_periods=72).mean().shift(1)
df['load_roll_7d_max']   = df['load_mw'].rolling(168, min_periods=72).max().shift(1)

# Load trend (current vs 24h ago)
df['load_trend_24h'] = df['load_mw'].shift(1) - df['load_mw'].shift(25)

# Renewable share lagged (availability 1h ago)
df['renewable_share_lag1'] = df['renewable_share'].shift(1)
df['wind_share_lag1']      = df['wind_share'].shift(1)

# Texas DC capacity feature (approximation: step function by year)
df['tx_dc_capacity_mw'] = df['year'].map(TX_DC_CAPACITY)
df['tx_dc_capacity_mw'] = df['tx_dc_capacity_mw'].fillna(45000)  # default for missing years

# Year trend (continuous)
df['year_trend'] = df['year'] - 2023

# Drop rows with NaN from lag/rolling features (first 48 hours)
df_model = df.dropna(subset=[c for c in df.columns if 'lag' in c or 'roll' in c or 'trend' in c]).copy()
print(f"    Model dataset: {len(df_model):,} rows after lag feature construction")

# Feature list for XGBoost
FEATURES = [
    'load_lag_1h', 'load_lag_6h', 'load_lag_12h', 'load_lag_24h', 'load_lag_48h',
    'load_roll_24h_mean', 'load_roll_24h_std', 'load_roll_7d_mean', 'load_roll_7d_max',
    'load_trend_24h',
    'renewable_share_lag1', 'wind_share_lag1',
    'hour', 'dow', 'month', 'is_weekend', 'is_summer', 'is_winter',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'tx_dc_capacity_mw', 'year_trend',
]
# Filter to features that actually exist
FEATURES = [f for f in FEATURES if f in df_model.columns]
TARGET = 'stressed'

print(f"    Features ({len(FEATURES)}): {FEATURES}")


### 4. TIME-SERIES CROSS-VALIDATION

In [ ]:
print("\n[4] Time-series walk-forward cross-validation...")

# With only 2 years of data (2023-2024), use monthly expanding window
# Train on months 1-N, test on month N+1

df_model = df_model.sort_values('dt').reset_index(drop=True)
df_model['period_month'] = df_model['dt'].dt.to_period('M')
months = sorted(df_model['period_month'].unique())

print(f"    Total months available: {len(months)}")
print(f"    Months: {[str(m) for m in months]}")

# Need at least 6 months training, then test on each subsequent month
MIN_TRAIN_MONTHS = 6
cv_results = []

all_oof_probs = []
all_oof_labels = []

for test_idx in range(MIN_TRAIN_MONTHS, len(months)):
    train_months = months[:test_idx]
    test_month   = months[test_idx]

    train = df_model[df_model['period_month'].isin(train_months)]
    test  = df_model[df_model['period_month'] == test_month]

    X_train = train[FEATURES].values
    y_train = train[TARGET].values
    X_test  = test[FEATURES].values
    y_test  = test[TARGET].values

    # Handle class imbalance via scale_pos_weight
    pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=pos_weight,
        random_state=SEED,
        use_label_encoder=False,
        eval_metric='logloss',
        verbosity=0,
    )
    model.fit(X_train, y_train, verbose=False)
    probs = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, probs) if y_test.sum() > 0 else np.nan

    all_oof_probs.extend(probs)
    all_oof_labels.extend(y_test)

    cv_results.append({
        'test_month':    str(test_month),
        'train_months':  len(train_months),
        'test_n':        len(test),
        'n_stressed':    int(y_test.sum()),
        'auc':           auc,
    })
    print(f"    Fold {test_idx - MIN_TRAIN_MONTHS + 1}: Train {len(train_months)}mo | "
          f"Test {test_month} | AUC={auc:.4f} | n_stressed={y_test.sum()}")

cv_df = pd.DataFrame(cv_results)
mean_auc = cv_df['auc'].mean()
print(f"\n    Mean AUC across {len(cv_df)} folds: {mean_auc:.4f}")
cv_df.to_csv(f"{OUT_DIR}/cv_results.csv", index=False)


### 5. FINAL MODEL ON ALL DATA

In [ ]:
print("\n[5] Fitting final model on full dataset...")

X_all = df_model[FEATURES].values
y_all = df_model[TARGET].values

# Use last 3 months as "holdout" for final evaluation
n_holdout_months = 3
holdout_months = months[-n_holdout_months:]
train_final = df_model[~df_model['period_month'].isin(holdout_months)]
test_final  = df_model[df_model['period_month'].isin(holdout_months)]

X_train_f = train_final[FEATURES].values
y_train_f = train_final[TARGET].values
X_test_f  = test_final[FEATURES].values
y_test_f  = test_final[TARGET].values

pos_weight_f = (y_train_f == 0).sum() / max((y_train_f == 1).sum(), 1)

final_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight_f,
    random_state=SEED,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=0,
)
final_model.fit(X_train_f, y_train_f)
y_prob = final_model.predict_proba(X_test_f)[:, 1]

# Find optimal threshold (maximize F1)
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores = [f1_score(y_test_f, (y_prob >= t).astype(int), zero_division=0) for t in thresholds]
optimal_thresh = thresholds[np.argmax(f1_scores)]
y_pred = (y_prob >= optimal_thresh).astype(int)

final_auc = roc_auc_score(y_test_f, y_prob)
final_ap  = average_precision_score(y_test_f, y_prob)

print(f"    Final model — holdout AUC: {final_auc:.4f}  |  Avg Precision: {final_ap:.4f}")
print(f"    Optimal threshold: {optimal_thresh:.2f}")
print(f"\n    Classification Report (threshold={optimal_thresh:.2f}):")
print(classification_report(y_test_f, y_pred, target_names=['Normal', 'Stressed']))


### 6. ROC AND PR CURVES

In [ ]:
print("\n[6] Plotting ROC and PR curves...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ROC curve
fpr, tpr, _ = roc_curve(y_test_f, y_prob)
ax1.plot(fpr, tpr, color='#e74c3c', linewidth=2.5, label=f'XGBoost (AUC = {final_auc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
ax1.fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("ROC Curve — ERCOT Grid Stress Classifier")
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# CV AUC scores
for i, row in cv_df.iterrows():
    ax1.axhline(row['auc'], color='gray', alpha=0.15, linewidth=1)
ax1.annotate(f"CV Mean AUC = {mean_auc:.3f}\n(walk-forward, 6-{len(months)-1} month windows)",
             xy=(0.55, 0.15), fontsize=9, color='gray',
             bbox=dict(boxstyle='round', fc='white', alpha=0.8))

# PR curve
precision, recall, _ = precision_recall_curve(y_test_f, y_prob)
ax2.plot(recall, precision, color='#2980b9', linewidth=2.5, label=f'XGBoost (AP = {final_ap:.3f})')
baseline = y_test_f.mean()
ax2.axhline(baseline, color='gray', linestyle='--', linewidth=1, label=f'Baseline (AP = {baseline:.3f})')
ax2.fill_between(recall, precision, alpha=0.1, color='#2980b9')
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curve — ERCOT Grid Stress Classifier\n(Critical: stressed hours are rare)")
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.suptitle("ERCOT Grid Vulnerability Classification — XGBoost Performance",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/roc_pr_curves.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/roc_pr_curves.png")


### 7. CONFUSION MATRIX

In [ ]:
print("\n[7] Confusion matrix...")

cm = confusion_matrix(y_test_f, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Predicted Normal', 'Predicted Stressed'],
            yticklabels=['Actual Normal', 'Actual Stressed'])
ax.set_title(f"Confusion Matrix (threshold = {optimal_thresh:.2f})\n"
             f"ERCOT Grid Stress Classifier | AUC = {final_auc:.3f}")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/confusion_matrix.png")


### 8. FEATURE IMPORTANCE

In [ ]:
print("\n[8] Feature importance plot...")

importance = pd.DataFrame({
    'feature':    FEATURES,
    'importance': final_model.feature_importances_,
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#e74c3c' if 'dc_capacity' in f or 'year' in f else
          '#2980b9' if 'renewable' in f or 'wind' in f else
          '#27ae60' if 'load_lag' in f or 'roll' in f else
          '#8e44ad' for f in importance['feature']]
bars = ax.barh(importance['feature'], importance['importance'], color=colors, alpha=0.85)
ax.set_xlabel("Feature Importance (XGBoost Gain)")
ax.set_title("ERCOT Grid Stress Classifier — Feature Importance\n"
             "Green=Load Lags  Blue=Renewables  Red=DC/Year  Purple=Time")
ax.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, importance['importance']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/feature_importance.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/feature_importance.png")

# SHAP values (if shap is available)
try:
    import shap
    print("\n    Computing SHAP values...")
    explainer   = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_test_f[:500])  # sample for speed

    fig, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(shap_values, X_test_f[:500], feature_names=FEATURES,
                      show=False, plot_type='bar')
    plt.title("SHAP Feature Importance — ERCOT Grid Stress Classifier")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/shap_importance.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {OUT_DIR}/shap_importance.png")

    # SHAP beeswarm (dot plot)
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test_f[:500], feature_names=FEATURES, show=False)
    plt.title("SHAP Beeswarm — Feature Impact on Grid Stress Prediction")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/shap_beeswarm.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {OUT_DIR}/shap_beeswarm.png")
except ImportError:
    print("    SHAP not installed — skipping SHAP plots (pip install shap)")


### 9. CV AUC OVER TIME PLOT

In [ ]:
print("\n[9] CV AUC over time...")

fig, ax = plt.subplots(figsize=(10, 4))
cv_df['test_month_dt'] = pd.to_datetime(cv_df['test_month'].astype(str))
ax.plot(cv_df['test_month_dt'], cv_df['auc'], 'o-', color='#e74c3c', linewidth=2, markersize=7)
ax.axhline(mean_auc, color='gray', linestyle='--', label=f'Mean AUC = {mean_auc:.3f}')
ax.axhline(0.8, color='#27ae60', linestyle=':', linewidth=1.5, label='Target AUC = 0.80')
ax.set_ylim(0.5, 1.01)
ax.set_xlabel("Test Month")
ax.set_ylabel("AUC-ROC")
ax.set_title("Walk-Forward CV AUC by Test Month\n(Training window expands left-to-right)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/cv_auc_over_time.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/cv_auc_over_time.png")


### 10. CALIBRATION PLOT

In [ ]:
print("\n[10] Calibration plot...")

prob_true, prob_pred = calibration_curve(y_test_f, y_prob, n_bins=10)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(prob_pred, prob_true, 's-', color='#e74c3c', linewidth=2, label='XGBoost')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives (Actual Stressed)")
ax.set_title("Calibration Plot — ERCOT Grid Stress Classifier")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/calibration_plot.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/calibration_plot.png")


### 11. SAVE METRICS

In [ ]:
metrics_df = pd.DataFrame([{
    'Model':            'XGBoost (ERCOT Stress Classifier)',
    'CV_Mean_AUC':      mean_auc,
    'Holdout_AUC':      final_auc,
    'Holdout_AP':       final_ap,
    'Optimal_Threshold': optimal_thresh,
    'N_train':          len(train_final),
    'N_test':           len(test_final),
    'Pct_Stressed':     y_all.mean(),
    'Features':         len(FEATURES),
    'CV_Folds':         len(cv_df),
}])
metrics_df.to_csv(f"{OUT_DIR}/model_metrics.csv", index=False)
print(f"\n    Saved: {OUT_DIR}/model_metrics.csv")


## Experiment 4: ERCOT Renewables

Source: `exp4_ercot_renewables.py`

EXP-4 FORMAL ANALYSIS: ERCOT Renewable Share vs Data Center Load Growth
Research Question:
As Texas AI data center capacity grew from ~45,000 MW (2023) to
~67,254 MW (2024), did renewable generation keep pace, or is new
DC load predominantly served by fossil fuels?

This is a key nuance for SQ2 (fossil fuel reliance). The naive narrative
is "more DCs = more fossil fuels." The more honest finding is that the
relationship is indirect: Texas added wind and solar alongside data centers,
but absolute fossil consumption also grew because total demand grew faster
than renewable build-out.

Analysis:
1. Monthly fuel mix trends (renewable share and absolute MWh)
2. Hourly renewable variability — does high DC load coincide with
low renewable availability? (the "coincidence of scarcity")
3. Hour-of-day fuel mix: when DCs run 24/7, what's serving them at 2am?
4. Load regression: does high load predict lower renewable share?
(When DCs push demand up, does renewable share fall — meaning fossil
units are called on to cover?)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

LOAD_PATH = "research_data/07_grid_stress/erco_load_hourly.csv"
FUEL_PATH = "research_data/07_grid_stress/erco_fuel_mix_hourly.csv"
OUT_DIR   = "results/exp4_ercot_renewables"

print("=" * 70)
print("EXP-4: ERCOT Renewable Share vs Data Center Load Growth")
print("=" * 70)


### 1. LOAD AND MERGE

In [ ]:
print("\n[1] Loading ERCOT data...")

df_load = pd.read_csv(LOAD_PATH)
df_load = df_load[df_load['type'] == 'D'].copy()
df_load['dt'] = pd.to_datetime(df_load['period'].str.replace('T', ' '), errors='coerce')
df_load = df_load.rename(columns={'demand_mw': 'load_mw'})[['dt', 'load_mw']].dropna()

df_fuel = pd.read_csv(FUEL_PATH)
df_fuel['dt'] = pd.to_datetime(df_fuel['period'].str.replace('T', ' '), errors='coerce')
df_fuel = df_fuel.dropna(subset=['dt'])

fuel_wide = df_fuel.pivot_table(index='dt', columns='fueltype', values='generation_mw', aggfunc='sum').fillna(0)
fuel_wide.columns = [f'gen_{c.lower()}' for c in fuel_wide.columns]

fuel_wide['total_gen']     = fuel_wide.sum(axis=1)
fuel_wide['wind']          = fuel_wide.get('gen_wnd', 0)
fuel_wide['solar']         = fuel_wide.get('gen_sun', 0)
fuel_wide['nat_gas']       = fuel_wide.get('gen_ng', 0)
fuel_wide['coal']          = fuel_wide.get('gen_col', 0)
fuel_wide['nuclear']       = fuel_wide.get('gen_nuc', 0)
fuel_wide['renewable_gen'] = fuel_wide['wind'] + fuel_wide['solar']
fuel_wide['fossil_gen']    = fuel_wide['nat_gas'] + fuel_wide['coal']
fuel_wide['renewable_share'] = fuel_wide['renewable_gen'] / fuel_wide['total_gen'].replace(0, np.nan)
fuel_wide['fossil_share']    = fuel_wide['fossil_gen'] / fuel_wide['total_gen'].replace(0, np.nan)

df = df_load.set_index('dt').join(fuel_wide, how='inner').reset_index()
df = df.rename(columns={'index': 'dt'})
df['dt'] = pd.to_datetime(df['dt'])
df['year']  = df['dt'].dt.year
df['month'] = df['dt'].dt.month
df['hour']  = df['dt'].dt.hour
df['date']  = df['dt'].dt.date
df = df.dropna(subset=['load_mw', 'renewable_share'])
print(f"    Merged: {len(df):,} rows | {df['dt'].min()} to {df['dt'].max()}")


### 2. MONTHLY FUEL MIX TRENDS

In [ ]:
print("\n[2] Monthly fuel mix trends...")

monthly = df.groupby(['year', 'month']).agg(
    renewable_share=('renewable_share', 'mean'),
    fossil_share=('fossil_share', 'mean'),
    wind_gwh=('wind', 'sum'),
    solar_gwh=('solar', 'sum'),
    nat_gas_gwh=('nat_gas', 'sum'),
    total_load_gwh=('load_mw', 'sum'),
    avg_load_mw=('load_mw', 'mean'),
).reset_index()
monthly['date'] = pd.to_datetime(monthly[['year', 'month']].assign(day=1))

fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)

# Panel 1: Renewable share
axes[0].plot(monthly['date'], monthly['renewable_share'] * 100,
             color='#27ae60', linewidth=2.5, marker='o', markersize=5)
axes[0].fill_between(monthly['date'], monthly['renewable_share'] * 100,
                     alpha=0.2, color='#27ae60')
axes[0].set_ylabel("Renewable Share (%)", fontsize=11)
axes[0].set_title("ERCOT Monthly Renewable Share (Wind + Solar)", fontsize=11)
axes[0].grid(alpha=0.3)
for yr in monthly.groupby('year')['renewable_share'].mean().items():
    color = '#1a8a4a' if yr[0] == 2024 else '#5dade2'
    axes[0].annotate(f"{yr[0]} avg: {yr[1]*100:.1f}%",
                     xy=(0.02 + (yr[0]-2023)*0.5, 0.92),
                     xycoords='axes fraction', color=color, fontsize=10)

# Panel 2: Absolute generation MWh
axes[1].fill_between(monthly['date'], monthly['wind_gwh']/1e6,
                     alpha=0.7, color='#2980b9', label='Wind (TWh)')
axes[1].fill_between(monthly['date'], monthly['solar_gwh']/1e6,
                     alpha=0.7, color='#f39c12', label='Solar (TWh)')
axes[1].set_ylabel("Generation (TWh/month)", fontsize=11)
axes[1].set_title("Wind and Solar Absolute Generation — Growing Despite DC Load", fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

# Panel 3: Average load (proxy for DC-driven baseline growth)
axes[2].plot(monthly['date'], monthly['avg_load_mw'] / 1000,
             color='#e74c3c', linewidth=2.5, marker='s', markersize=5)
axes[2].set_ylabel("Avg Load (GW)", fontsize=11)
axes[2].set_title("ERCOT Average Load — Structural Upward Shift 2023→2024", fontsize=11)
axes[2].grid(alpha=0.3)

plt.suptitle("ERCOT Fuel Mix and Load Trends (2023-2024)\n"
             "AI Data Center Capacity: 45,000 MW (2023) → 67,254 MW (2024)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/ercot_monthly_fuel_trends.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/ercot_monthly_fuel_trends.png")

# Year-over-year stats
for yr in [2023, 2024]:
    subset = monthly[monthly['year'] == yr]
    print(f"    {yr}: Avg renewable share = {subset['renewable_share'].mean()*100:.1f}%  "
          f"| Avg load = {subset['avg_load_mw'].mean()/1000:.1f} GW")


### 3. HOUR-OF-DAY FUEL MIX

In [ ]:
print("\n[3] Hourly fuel mix profiles...")

hourly_avg = df.groupby('hour').agg(
    renewable_share=('renewable_share', 'mean'),
    fossil_share=('fossil_share', 'mean'),
    wind_mw=('wind', 'mean'),
    solar_mw=('solar', 'mean'),
    nat_gas_mw=('nat_gas', 'mean'),
    load_mw=('load_mw', 'mean'),
).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Fuel shares by hour
ax1.fill_between(hourly_avg['hour'], hourly_avg['renewable_share']*100,
                 alpha=0.7, color='#27ae60', label='Renewable share (%)')
ax1.fill_between(hourly_avg['hour'], hourly_avg['fossil_share']*100,
                 alpha=0.5, color='#e74c3c', label='Fossil share (%)')
ax1.set_xlabel("Hour of Day")
ax1.set_ylabel("Generation Share (%)")
ax1.set_title("ERCOT Average Fuel Mix by Hour\n(AI data centers run 24/7 — what's powering them at 3am?)")
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_xticks(range(0, 24, 3))

# Right: MW by fuel type
ax2.plot(hourly_avg['hour'], hourly_avg['wind_mw']/1000, color='#2980b9',
         linewidth=2.5, label='Wind (GW)')
ax2.plot(hourly_avg['hour'], hourly_avg['solar_mw']/1000, color='#f39c12',
         linewidth=2.5, label='Solar (GW)')
ax2.plot(hourly_avg['hour'], hourly_avg['nat_gas_mw']/1000, color='#e74c3c',
         linewidth=2, linestyle='--', label='Natural Gas (GW)')
ax2.plot(hourly_avg['hour'], hourly_avg['load_mw']/1000, color='black',
         linewidth=1.5, linestyle=':', label='Total Load (GW)')
ax2.set_xlabel("Hour of Day")
ax2.set_ylabel("Generation (GW)")
ax2.set_title("ERCOT Average Hourly Generation by Fuel Type\n(Wind peaks overnight, solar peaks at noon)")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
ax2.set_xticks(range(0, 24, 3))

# Overnight annotation
ax2.axvspan(0, 6, alpha=0.07, color='navy')
ax2.annotate("Overnight: DC 24/7 load\ncovered by wind + gas",
             xy=(3, hourly_avg.loc[3, 'load_mw']/1000 + 1.5),
             fontsize=8.5, color='navy', ha='center',
             bbox=dict(boxstyle='round', fc='#eef', ec='navy', alpha=0.8))

plt.suptitle("ERCOT Fuel Mix by Hour of Day — The Overnight DC Load Question",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/ercot_hourly_fuel_profile.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/ercot_hourly_fuel_profile.png")

overnight_ren = hourly_avg[hourly_avg['hour'].between(0, 5)]['renewable_share'].mean()
midday_ren    = hourly_avg[hourly_avg['hour'].between(10, 14)]['renewable_share'].mean()
print(f"    Overnight (0-5h) avg renewable share: {overnight_ren*100:.1f}%")
print(f"    Midday (10-14h) avg renewable share:  {midday_ren*100:.1f}%")


### 4. LOAD VS RENEWABLE SHARE REGRESSION

In [ ]:
print("\n[4] Does high load reduce renewable share? (fossil unit dispatch)")

df['load_zscore']  = (df['load_mw'] - df.groupby('month')['load_mw'].transform('mean')) / \
                     df.groupby('month')['load_mw'].transform('std')
df['month_str'] = df['month'].astype(str)

model = smf.ols(
    "renewable_share ~ load_zscore + C(hour) + C(month_str) + C(year)",
    data=df
).fit()

coef_load = model.params.get('load_zscore', np.nan)
pval_load = model.pvalues.get('load_zscore', np.nan)
print(f"    Load → Renewable Share: coef = {coef_load:.5f}  p = {pval_load:.6f}")
print(f"    R-squared: {model.rsquared:.4f}")

direction = "DECREASES" if coef_load < 0 else "INCREASES"
print(f"    Interpretation: A 1-SD increase in (within-month) load {direction} "
      f"renewable share by {abs(coef_load)*100:.2f} percentage points.")

# Scatter: load percentile vs renewable share
df['load_pct_bin'] = pd.qcut(df['load_mw'], 20, labels=False)
bin_avg = df.groupby('load_pct_bin')[['load_mw', 'renewable_share']].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(bin_avg['load_mw']/1000, bin_avg['renewable_share']*100,
           s=80, color='#8e44ad', alpha=0.8, zorder=5)
# Trend line
z = np.polyfit(bin_avg['load_mw'], bin_avg['renewable_share']*100, 1)
p = np.poly1d(z)
x_line = np.linspace(bin_avg['load_mw'].min(), bin_avg['load_mw'].max(), 100)
ax.plot(x_line/1000, p(x_line), color='#e74c3c', linewidth=2,
        label=f"Trend (coef={coef_load*100:.3f} pp per SD, p={pval_load:.4f})")
ax.set_xlabel("ERCOT Load (GW)", fontsize=11)
ax.set_ylabel("Renewable Generation Share (%)", fontsize=11)
ax.set_title("ERCOT: High Load → Lower Renewable Share\n"
             "(When AI DCs push demand up, fossil units make up the difference)")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/load_vs_renewable_share.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/load_vs_renewable_share.png")


### 5. YEAR-OVER-YEAR ABSOLUTE LOAD GROWTH

In [ ]:
print("\n[5] Year-over-year absolute load and generation comparison...")

yr_stats = df.groupby('year').agg(
    total_load_twh=('load_mw', lambda x: x.sum() / 1e6),
    total_renewable_twh=('renewable_gen', lambda x: x.sum() / 1e6),
    total_fossil_twh=('fossil_gen', lambda x: x.sum() / 1e6),
    avg_renewable_share=('renewable_share', 'mean'),
    avg_load_gw=('load_mw', lambda x: x.mean() / 1e3),
    peak_load_gw=('load_mw', lambda x: x.max() / 1e3),
    n_stressed_hours=('load_mw', lambda x: (x >= x.quantile(0.90)).sum()),
).reset_index()

print(yr_stats.to_string(index=False))
yr_stats.to_csv(f"{OUT_DIR}/year_over_year_stats.csv", index=False)
print(f"    Saved: {OUT_DIR}/year_over_year_stats.csv")

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
x = yr_stats['year'].astype(str)

# Stacked bar: absolute renewable vs fossil
axes[0].bar(x, yr_stats['total_renewable_twh'], color='#27ae60', alpha=0.85, label='Renewable (TWh)')
axes[0].bar(x, yr_stats['total_fossil_twh'], bottom=yr_stats['total_renewable_twh'],
            color='#e74c3c', alpha=0.75, label='Fossil (TWh)')
axes[0].set_ylabel("Electricity Generated (TWh/yr)")
axes[0].set_title("ERCOT Annual Generation\nby Fuel Type")
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Renewable share
axes[1].bar(x, yr_stats['avg_renewable_share']*100, color='#2ecc71', alpha=0.85)
axes[1].set_ylabel("Avg Renewable Share (%)")
axes[1].set_title("Average Annual\nRenewable Share")
for i, (yr, val) in enumerate(zip(yr_stats['year'], yr_stats['avg_renewable_share'])):
    axes[1].text(i, val*100+0.5, f"{val*100:.1f}%", ha='center', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

# Stressed hours
axes[2].bar(x, yr_stats['n_stressed_hours'], color='#e74c3c', alpha=0.85)
axes[2].set_ylabel("Hours at >= 90th Pct Load")
axes[2].set_title("Annual Stressed\nHours (>= P90 Load)")
for i, (yr, val) in enumerate(zip(yr_stats['year'], yr_stats['n_stressed_hours'])):
    axes[2].text(i, val+5, str(val), ha='center', fontsize=11)
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle("ERCOT 2023 vs 2024: Renewables Growing, But So Is Load and Stress",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/year_over_year_comparison.png", dpi=150)
plt.close()
print(f"    Saved: {OUT_DIR}/year_over_year_comparison.png")
